# Paper 5, RB-7 - the last rebuttal run: does the RoPE story hold beyond ViT-B at 224, and is the head null fair?

**Naming.** W1-W7 = September runs. RB = rebuttal runs: RB-0 single heads (`w8_*`, canonical = cuDNN 9.10.2 re-run),
RB-1/1b drift checks, RB-2 axis map, RB-3 intact/shuffled/null, RB-4 mixed RoPE, RB-5 window edge, RB-6 shared
erasure. This is **RB-7**; it writes to `./rb7_output/`. Five parts, switched by `RB7_PARTS` (default `d,a,b,e,c`,
run in that order: cheapest and most important first, the riskiest last).

| part | question (reviewer) | models | what it adds to the paper |
|---|---|---|---|
| **d** | is "the six top SSDC heads cost more than random sets" only a block-position effect? (91UE W4, RxJ2 Q2) | naver ViT-B, ViT-S, reg1-gap (+ DINOv3, SSDC only) | M20 sentence, Table 21 "Top 6 vs random" |
| **a** | do the rebuilding heads keep their importance and axis at 320×320? (91UE Q4, W5) | naver ViT-B, reg1-gap, naver ViT-S | Appendix H.3 |
| **b** | does the story hold in a larger model? (91UE W5, RxJ2 W2) | naver ViT-L/16 (24 blocks, 16 heads) | new Appendix H.4 |
| **e** | do the axes compete for attention? what does the hybrid's classifier get from the rotations? (NpkG Q2, RxJ2 Q3) | 4 RoPE + hybrid | Appendix E.1 and G.2 sentences |
| **c** | does it hold in a RoPE encoder trained for a VLM? (RxJ2 W2) | Qwen2-VL-2B vision encoder (675M, 32 blocks) | new Appendix H.5 |

Every part repeats, in the new setting, the paper's own RoPE tests: (1) removing one axis's rotation planes in every
block removes that axis and spares the other (Figure 4a); (2) after rotations are withheld from the first half of
the blocks and restored, later blocks rebuild position (Figure 4b); (3) single heads in the first blocks after the
window do the rebuilding, mostly one axis each (Figure 4d), and the costliest ones matter for top-1 (Figure 5b).
Every reading is fixed in the CONFIG cell before the run; the analysis prints a verdict per question and writes
`rb7_report.md`, which states the result and the exact paper text for each outcome.

**Coherence.** Same environment (`env_paper5_cudnn910`, checked by the pre-flight), datasets (repack, 1,000 images
for SSDC and probes; ILSVRC, 5,000 images for top-1), batch 256, seed 20260908, fp32, RPI, readouts (input of each
block after its first LayerNorm; FINAL = output of the last block through the final norm) and probe protocol as
RB-0 to RB-6. Anchors, all bit for bit: RB-0 parts A, C, D and RB-3 (part d); W6's 320×320 runs (part a); W5's
attention statistics (part e). ViT-L and Qwen2-VL have no earlier run; they get internal gates only.

In [ ]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas", "transformers"]:
    ensure(p)
ensure("huggingface_hub")
print("environment ready")

In [ ]:
import os, json, time, warnings, traceback, shutil, hashlib, platform, socket, gc
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.executable)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# config
REPO_PATH  = os.environ.get("RB7_REPO", "")
RB0_DIR    = os.environ.get("RB7_RB0_DIR", "")       # canonical RB-0 (rb0_cudnn910/w8_output)
RB2_DIR    = os.environ.get("RB7_RB2_DIR", "")       # RB-2 output folder (rb2_output)
RB3_DIR    = os.environ.get("RB7_RB3_DIR", "")       # RB-3 output folder (rb3_output)
PARTS      = [p for p in os.environ.get("RB7_PARTS", "d,a,b,e,c").split(",") if p]
RESUME     = os.environ.get("RB7_RESUME", "1") == "1"
STRICT     = os.environ.get("RB7_STRICT", "1") == "1"
TEST_MODE  = os.environ.get("RB7_TEST", "0") == "1"   # CPU smoke test on tiny random models, not for results
OUT_DIR    = Path(os.environ.get("RB7_OUT", "rb7_output"))
QWEN_ID    = os.environ.get("RB7_QWEN", "Qwen/Qwen2-VL-2B-Instruct")

N_SSDC, N_ACC, BATCH_SIZE = 1000, 5000, 256
METRIC, TORCH_SEED = "manhattan", 20260908
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"
ACC_DATASET  = "ILSVRC/imagenet-1k"
PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8   # W2-W6 / RB-0 / RB-2 protocol, unchanged
PROBE_CHUNK, PROBE_DEVICE = 8192, None
NOISE_SEEDS, NOISE_BOOT = [0, 1, 2], 200                                       # W5 / RB-0 / RB-2 probe-noise recipe
EXACT_TOL = 0.0
EXPECT_TORCH, EXPECT_CUDNN = "2.10.0+cu128", 91002
EXPECT_CUDNN_PKGS = {"nvidia-cudnn-cu12": "9.10.2.21"}
FIRST_LAYER_910 = {"naver": 0.00041303808169361526, "DINOv3": 0.0004745740108653513,
                   "small": 0.000510305810057617, "reg1_gap": 0.00026782853342761795}
FIRST_LAYER_RANGE = (1e-4, 2e-3)      # TF32-level patch embedding, as for every model RB-1b / RB-6 measured
REPLAY_CHECK_N = 512

MODEL_SPECS = {
    "naver":    dict(name="vit_base_patch16_rope_224.naver_in1k",        classifier=True,  label="naver ViT-B"),
    "DINOv3":   dict(name="vit_base_patch16_dinov3.lvd1689m",            classifier=False, label="DINOv3 ViT-B"),
    "small":    dict(name="vit_small_patch16_rope_224.naver_in1k",       classifier=True,  label="naver ViT-S"),
    "reg1_gap": dict(name="vit_base_patch16_rope_reg1_gap_256.sbb_in1k", classifier=True,  label="reg1-gap ViT-B"),
    "rope_ape": dict(name="vit_base_patch16_rope_ape_224.naver_in1k",    classifier=True,  label="hybrid ViT-B"),
    "large":    dict(name="vit_large_patch16_rope_224.naver_in1k",       classifier=True,  label="naver ViT-L"),
}

# part d: block-matched null for the six top SSDC heads (RB-3's set)
D_MODELS = ["naver", "small", "reg1_gap", "DINOv3"]
D_WINDOW, D_POST = list(range(0, 6)), list(range(6, 12))
D_N_NULL, D_SEED = 48, 20261109
D_N_RAND_GATE = 2                     # RB-0 part D one-head-per-block sets re-measured as a reproducibility gate
D_PERM_N = 10000
# part a: 320 x 320
A_MODELS, A_RES = ["naver", "reg1_gap", "small"], 320
A_REF_HEAD = {"naver": (6, 6), "reg1_gap": (6, 9), "small": (7, 4)}     # the heads Section 4.5 names (costliest at 224)
# part b: ViT-L
B_WINDOWS = {"w0_5": list(range(0, 6)), "w0_11": list(range(0, 12)), "w0_17": list(range(0, 18))}
B_MAIN = "w0_11"                      # first half of 24 blocks, as 0-5 is in ViT-B
# parts a/b: axis test and single heads
N_AXIS_TOP = 5                        # axis map for the top 5 heads by top-1 cost and the top 5 by SSDC (union)
RAND_PLANES_SEED = 16                 # 16 random planes of 32 (Figure 4a's "rand." control), drawn once per model
# part e: attention statistics (W5 part E recipe)
E_MODELS = ["naver", "DINOv3", "small", "reg1_gap", "rope_ape"]
E_IMAGES, E_BATCH = 64, 32            # W5 part E: first 64 repack images, natural order, batch 32
# part c: Qwen2-VL vision encoder
C_RES, C_BATCH = 224, 128             # 224 x 224 -> a 16 x 16 grid of 14-px patches; batch 128 (no anchor to match)
C_WINDOWS = {"w0_7": list(range(0, 8)), "w0_15": list(range(0, 16)), "w0_23": list(range(0, 24))}
C_MAIN = "w0_15"
C_AXIS_READOUT = 11                   # one third of 32 blocks (block 4 of 12 in ViT-B; block 8 of 24 in ViT-L)
C_PROBES_BASE = [4, 11, 16, 31]       # + FINAL

# the readings, fixed before the run
AXIS_GONE, AXIS_KEPT = 0.10, 0.50
REBUILD_HI, REBUILD_LO = 0.50, 0.20
RULES = {
    "D_NULL": (f"Part d. Null: {D_N_NULL} random head sets per model with exactly the six top SSDC heads' block counts "
               "(heads drawn uniformly within each block, the top-6 set itself excluded; seed fixed). Cost = top-1 points "
               "lost against the 0-5 window (classifiers) and SSDC lost entering block 11 against Cref (all four). "
               f"HOLDS if the top-6 cost exceeds all {D_N_NULL} sets; HOLDS AT 95% if it exceeds at least "
               f"{int(np.ceil(0.95 * D_N_NULL))}; DOES NOT HOLD otherwise."),
    "D_WITHIN": ("Part d, from RB-0 (no new forward passes): Spearman of single-head SSDC cost vs top-1 cost computed "
                 "within each of blocks 6-10 and averaged; p from a permutation that shuffles top-1 costs within blocks."),
    "AXIS": (f"Axis test (parts a, b, c): retention = (arm - floor) / (intact - floor) of the row and column probes at "
             f"the axis readout. AXES SEPARATE if removing the row planes in every block leaves row retention <= "
             f"{AXIS_GONE} and column retention >= {AXIS_KEPT}, and removing the column planes does the converse; "
             "PARTLY otherwise if the removed axis always falls more than the other; NOT SEPARATE if not."),
    "REBUILD": (f"Rebuild (parts a, b, c): normalised SSDC entering the last block after the first-half window "
                f"(0: floor, 1: intact). REBUILDS if >= {REBUILD_HI} (ViT-B at 224: 0.50-0.71); REBUILDS LESS if "
                f"{REBUILD_LO}-{REBUILD_HI}; DOES NOT REBUILD below {REBUILD_LO}."),
    "HEADS": ("Single heads (parts a, b): every head after the window, rotations off while the rest recover. SSDC rank "
              "= RB-2's rule (blocks before the last: SSDC lost entering the last block; last block: at FINAL); top-1 "
              f"cost on 5,000 images with paired SE. Axis map (RB-2's windows and noise bar) for the union of the top "
              f"{N_AXIS_TOP} by top-1 and the top {N_AXIS_TOP} by SSDC (part a adds the two costliest heads at 224)."),
    "A_KEEPS": ("Part a: the head Section 4.5 names ((6,6), (6,9), (7,4)) KEEPS ITS RANK if it is among the 3 costliest "
                "single heads by top-1 at 320, and KEEPS ITS AXIS if its RB-2 label at 320 equals its label at 224."),
    "B_HEADS": ("Part b: top SSDC head and costliest top-1 head; whether each sits in the first two blocks after the "
                "window (12-13); share of the summed positive top-1 cost in blocks 12-13; Spearman of SSDC vs top-1 cost "
                "over all 192 heads; how many mapped heads favour one axis."),
    "E1": ("Part e (NpkG Q2): mean attention distance along rows |dr| and along columns |dc| (patches), blocks 1-4, "
           "head-averaged, 64 natural images. WIDER POOLING SUPPORTED if removing an axis's planes widens attention "
           "along that axis in at least 7 of the 8 (model, axis) cases and the widening rank-correlates (Spearman > 0.5, "
           "descriptive) with the rise of the other axis's probe above intact at block 4 (W5); NOT SUPPORTED if it widens "
           "in at most 4 of 8; MIXED otherwise."),
    "E2": ("Part e (RxJ2 Q3): hybrid, blocks 4-11, head-averaged distance and entropy. ROTATIONS KEEP LATE ATTENTION "
           "LOCAL if removing the rotations raises the mean distance in at least 6 of the 8 blocks and by at least 0.25 "
           "patches on average, while removing the embedding changes it by less than half as much; otherwise NOT SHOWN."),
    "C_START": ("Part c: STARTS NEAR ZERO if SSDC entering block 0 is at most 0.05 (Figure 2's RoPE pattern: ViT-B RoPE models "
                "0.027-0.028, APE models 0.42-0.57), i.e. spatial structure is built by attention, not supplied by an embedding."),
}

if TEST_MODE:
    N_SSDC, N_ACC, BATCH_SIZE = 24, 32, 8
    PROBE_PASSES, NOISE_BOOT, D_PERM_N, D_N_NULL = 2, 10, 200, 6
    E_IMAGES, E_BATCH, C_BATCH = 16, 8, 8
    OUT_DIR = Path(os.environ.get("RB7_OUT", "rb7_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)
_bad = [p for p in PARTS if p not in ("d", "a", "b", "e", "c")]
if _bad: raise SystemExit(f"Unknown entries in RB7_PARTS: {_bad}")
PARTS = [p for p in ("d", "a", "b", "e", "c") if p in PARTS]

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"rb7_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return bool(passed) or skip
def scope_failed(scope):
    return [g for g in GATES if g["scope"] == scope and g["status"] == "FAIL"]
print(f"device {DEVICE} | fp32 | parts {PARTS}")
print(f"SSDC/probes: {SSDC_DATASET} x {N_SSDC} | top-1: {ACC_DATASET} x {N_ACC} | batch {BATCH_SIZE} | seed {TORCH_SEED}")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | strict {STRICT} | test mode {TEST_MODE} | run {RUN_STAMP}")

## Pre-flight: are we in `env_paper5_cudnn910`?
Same checks as RB-2 to RB-6. Nothing is measured if any fails.

In [ ]:
def _sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as ex:
        return f"(failed: {ex!r})"

def _loaded_cudnn():
    try:
        with open("/proc/self/maps") as f:
            return sorted({l.split()[-1] for l in f if "libcudnn" in l and l.split()[-1].startswith("/")})
    except Exception:
        return []

import importlib.metadata as _md
if DEVICE == "cuda":
    _x = torch.randn(2, 3, 32, 32, device="cuda"); _w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(_x, _w, stride=16); torch.cuda.synchronize(); del _x, _w
CUDNN_PKGS = {}
for _d in _md.distributions():
    _n = (_d.metadata.get("Name") or "").lower()
    if "cudnn" in _n: CUDNN_PKGS[_n] = _d.version
import transformers as _tf
ENV = {
    "python": sys.executable, "prefix": sys.prefix, "host": socket.gethostname(), "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "transformers": _tf.__version__, "numpy": np.__version__,
    "cudnn_reported": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "libcudnn_loaded": _loaded_cudnn(), "cudnn_packages": CUDNN_PKGS,
    "flags": {"matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
              "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
              "float32_matmul_precision": torch.get_float32_matmul_precision()},
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu": _sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv") if DEVICE == "cuda" else "cpu",
}
(OUT_DIR / f"rb7_pip_freeze_{RUN_STAMP}.txt").write_text(_sh(f"{sys.executable} -m pip freeze", timeout=300))
(OUT_DIR / f"rb7_environment_{RUN_STAMP}.json").write_text(json.dumps(ENV, indent=1, default=str))
print(json.dumps({k: v for k, v in ENV.items() if k != "gpu"}, indent=1, default=str)); print(ENV["gpu"])

_pre = []
if TEST_MODE:
    gate("pre-flight (cuDNN 9.10.2 environment)", "env", True, "test mode on CPU", skip=True)
else:
    _pre.append(gate("CUDA available", "env", DEVICE == "cuda"))
    _pre.append(gate(f"torch is {EXPECT_TORCH}", "env", torch.__version__ == EXPECT_TORCH, torch.__version__))
    _pre.append(gate(f"PyTorch reports cuDNN {EXPECT_CUDNN} (9.10.2)", "env", ENV["cudnn_reported"] == EXPECT_CUDNN,
                     str(ENV["cudnn_reported"])))
    _prefix = os.path.realpath(sys.prefix)
    _libs = ENV["libcudnn_loaded"]
    _pre.append(gate("every loaded libcudnn file comes from this environment's nvidia/cudnn folder", "env",
                     bool(_libs) and all(os.path.realpath(l).startswith(_prefix) and "/nvidia/cudnn/" in l for l in _libs),
                     "; ".join(Path(l).name for l in _libs) + f" (prefix {_prefix})"))
    _pre.append(gate("the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13)", "env",
                     CUDNN_PKGS == EXPECT_CUDNN_PKGS, str(CUDNN_PKGS)))
    _f = ENV["flags"]
    _pre.append(gate("TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0)", "env",
                     _f["matmul_allow_tf32"] is False and _f["cudnn_allow_tf32"] is True and _f["cudnn_benchmark"] is False
                     and _f["cudnn_deterministic"] is False and _f["float32_matmul_precision"] == "highest", str(_f)))
PREFLIGHT_OK = all(_pre)
if not PREFLIGHT_OK:
    raise SystemExit("PRE-FLIGHT FAILED: this notebook must run in env_paper5_cudnn910 (cuDNN 9.10.2), with PyTorch's "
                     "default flags. Nothing was measured. Switch the kernel / python and run again.")
print("\npre-flight OK")

In [ ]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB7_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
REPO_COMMIT = _sh(f"git -C {repo} rev-parse HEAD")
print("repo root:", repo, "| commit", REPO_COMMIT)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope, load_ape, load_ape_timm
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

## Earlier results RB-7 is gated against (fingerprint check)
Every file is checked by sha256 before anything is measured. RB-0 = the **cuDNN 9.10.2 re-run**
(`rb0_cudnn910/w8_output`); RB-2 and RB-3 = their output folders; W5 and W6 = the committed September files in the
repo checkout. Only the files the requested parts need are required.

In [ ]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

SHA = {   # name -> sha256
    # RB-0 (canonical cuDNN 9.10.2 re-run, 4 Nov)
    "w8_partA_naver.json":      "f50b694d24279ce63b8dac8da1906b7e60ab6522ce8de4c992b1c87ceed10914",
    "w8_partA_DINOv3.json":     "f1d963652ac3867c79bb170da2b2dd86ff8e978008f5ff59304a69947bb909b5",
    "w8_partA_small.json":      "be01ed6297ef819a8fc219853c07fc9a81a76d2c6ffbe9da8501379fdf6acbbb",
    "w8_partA_reg1_gap.json":   "da5ae49a2abc21aa0df8f4c43b249727a7edfe444254e239e637ed96a312fefb",
    "w8_partC_naver.json":      "c983414404ca645bc8144a2bd4ec76bfb583f4c5c9588ec8faffe853e5c3b985",
    "w8_partC_small.json":      "bf895274486dfaa9ab042118e15bb7c9863a5f6201ab27ec11fb539ecf7126f6",
    "w8_partC_reg1_gap.json":   "8b5641235e05b20bfdc597032097b2f85878c228bdc32ec8b63888227e393698",
    "w8_perimage_naver.npz":    "5fe4a53ecbf99bc583c56ef6ec9f774c5bd6027e1734dce16fe0c230c36358c6",
    "w8_perimage_small.npz":    "b9f6a3833766b73618c82b3e5ecbb81af672b11eaa49fab503e27bb8734cf40b",
    "w8_perimage_reg1_gap.npz": "10310f0c0109d65f0b594f7774006ddd2c69f5fc74438b13d32956d39738643e",
    "w8_partD_naver.json":      "4b385c6bfde78d76a51276108fd9de741a394db3ccdb56980018a8ac1a885836",
    "w8_partD_DINOv3.json":     "6a49100079cfe304f1e111b671d11f61c1c4763d55280f0b37c18dfd768c11c0",
    "w8_partD_small.json":      "5de620a0ea29fa08c29dc56c87e45baabdf111781c79bf69191180e664e29fd0",
    "w8_partD_reg1_gap.json":   "85d72d98fff492d5ff72955169e4739e5f0b8b96d73c11ffb794f9f02ed68362",
    # RB-2 (axis map at 224) and RB-3 (top-6 set, top-1 per image)
    "rb2_axis_map.csv":         "81ba8d582391f2b4852b3b71c3d598b282f628ff0a52a14fdb12d86a5db96559",
    "rb3_naver.json":           "fbd214f2b3cfa5f9d09364d19883a4c858427861ebffe227c5171226265728d2",
    "rb3_small.json":           "061077575fff99f8906876bdd6d56b036c20276203b8525c8ab88be634638757",
    "rb3_reg1_gap.json":        "f4905b047b846498755c1764fc7601ef1d16179f3e62f0720907b4ac267dc397",
    "rb3_DINOv3.json":          "22ccbd911681d7cb6ec77b3780b3233aa52b7024b4c5281f0c3d797a4d8dc997",
    "rb3_perimage_naver.npz":   "defe7f13b3cf0fa3f7efcf10a99f4af762f435309896afddc2d365dd3384f0f5",
    "rb3_perimage_small.npz":   "e859c9955f1cc903d8473bf4a79cbe2433411a616d1f181611d362b2efcf57be",
    "rb3_perimage_reg1_gap.npz": "234725b955ad3a17dbb8f1f5848c0a395dadf44a9a7ae1448734355ad409a6da",
    # committed September files (repo checkout, commit 28e0856)
    "w5/w5_partA_naver_2026-09-13T10-16-41Z.json":    "2df4ab2b5ab7d827528d77896018ef6d5bb8cb0abe13b6dbfd77ad9078423fa6",
    "w5/w5_partA_DINOv3_2026-09-13T10-16-41Z.json":   "94a66af69c26eff4a55f8f0a80919d75a839461820d7cfaa853f463e8761d78f",
    "w5/w5_partA_small_2026-09-13T10-16-41Z.json":    "f7eb94b16a2660e7cac9e19076c68e321507b865b74a1d6b0e0a47925856e3d8",
    "w5/w5_partA_reg1_gap_2026-09-13T10-16-41Z.json": "ec2e79b18f94ff36d91a7495c78d882ef042bf75d1fb421bf6faa457f438727a",
    "w5/w5_partE_attention_2026-09-13T10-16-41Z.json": "a40aaada6a7ab2ccbaef04f81a87609d54143015edbf45c2949dadc40becb180",
    "w6/w6_partA_naver_2026-09-14T07-09-56Z.json":    "36f467c1092d340ff36fe0517424dcc55aaa02b1c32195809d5a020108e16b83",
    "w6/w6_partA_reg1_gap_2026-09-14T07-09-56Z.json": "5825f5c035a06f067aa4cc039e09c201f00c6b5be832681615d487cf57ae9851",
    "w6/w6_partA_small_2026-09-14T07-09-56Z.json":    "ede2f69e17326ff37d446755cc006acaf0387f9bfc3d4eb9ef7a13271337ea10",
}
W5_PARTA = {"naver": "w5/w5_partA_naver_2026-09-13T10-16-41Z.json", "DINOv3": "w5/w5_partA_DINOv3_2026-09-13T10-16-41Z.json",
            "small": "w5/w5_partA_small_2026-09-13T10-16-41Z.json", "reg1_gap": "w5/w5_partA_reg1_gap_2026-09-13T10-16-41Z.json"}
W5_PARTE = "w5/w5_partE_attention_2026-09-13T10-16-41Z.json"
W5_KEY = {"naver": "RoPE · naver", "DINOv3": "RoPE · DINOv3", "small": "RoPE · small", "reg1_gap": "RoPE · reg1_gap",
          "rope_ape": "RoPE · rope_ape"}
W6_PARTA = {"naver": "w6/w6_partA_naver_2026-09-14T07-09-56Z.json", "reg1_gap": "w6/w6_partA_reg1_gap_2026-09-14T07-09-56Z.json",
            "small": "w6/w6_partA_small_2026-09-14T07-09-56Z.json"}
RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"

NEED = {}   # folder -> [file names]
if "d" in PARTS or "a" in PARTS:
    NEED.setdefault("RB0", [])
    for k in (set(D_MODELS if "d" in PARTS else []) | set(A_MODELS if "a" in PARTS else [])):
        NEED["RB0"].append(f"w8_partA_{k}.json")
        if MODEL_SPECS[k]["classifier"]: NEED["RB0"] += [f"w8_partC_{k}.json", f"w8_perimage_{k}.npz"]
        if "d" in PARTS and k in D_MODELS: NEED["RB0"].append(f"w8_partD_{k}.json")
    NEED["RB0"] = sorted(set(NEED["RB0"]))
if "a" in PARTS: NEED["RB2"] = ["rb2_axis_map.csv"]
if "d" in PARTS:
    NEED["RB3"] = [f"rb3_{k}.json" for k in D_MODELS] + [f"rb3_perimage_{k}.npz" for k in D_MODELS if MODEL_SPECS[k]["classifier"]]
if "a" in PARTS: NEED.setdefault("REPO", []).extend(W6_PARTA[k] for k in A_MODELS)
if "e" in PARTS: NEED.setdefault("REPO", []).extend([W5_PARTA[k] for k in E_MODELS if k in W5_PARTA] + [W5_PARTE])
FOLDERS = {"RB0": RB0_DIR, "RB2": RB2_DIR, "RB3": RB3_DIR, "REPO": str(RES_ROOT)}
ENVNAME = {"RB0": "RB7_RB0_DIR", "RB2": "RB7_RB2_DIR", "RB3": "RB7_RB3_DIR", "REPO": "RB7_REPO"}
FILES, _bad_fp = {}, []
for fol, names in NEED.items():
    base = Path(FOLDERS[fol]).expanduser() if FOLDERS[fol] else None
    if base is None and not TEST_MODE:
        raise SystemExit(f"Set {ENVNAME[fol]} (needed by parts {PARTS}). Nothing was measured.")
    for fn in names:
        if TEST_MODE and fol == "REPO":
            gate(f"{fn} loaded", fol, True, "test mode: tiny models have no September anchors", skip=True); continue
        p = (base / fn) if base else None
        if p is None or not p.exists():
            gate(f"{fn} present", fol, False, str(p)); _bad_fp.append(fn); continue
        FILES[fn] = p
        if TEST_MODE:
            gate(f"{fn} is the expected file", fol, True, "test mode: hash not checked", skip=True)
        else:
            d = sha256_file(p)
            if not gate(f"{fn} is the expected file (sha256)", fol, d == SHA[fn], f"{d[:16]}... (expected {SHA[fn][:16]}...)"):
                _bad_fp.append(fn)
if _bad_fp:
    raise SystemExit(f"FINGERPRINT FAILED for {_bad_fp}: check RB7_RB0_DIR (must be rb0_cudnn910/w8_output), RB7_RB2_DIR, "
                     "RB7_RB3_DIR and the repo checkout. Nothing was measured.")
def jload(fn): return json.loads(FILES[fn].read_text())
def nload(fn): return {n: z_[n] for z_ in [np.load(FILES[fn])] for n in z_.files}
RB0, RB3 = {}, {}
for k in MODEL_SPECS:
    if f"w8_partA_{k}.json" in FILES:
        RB0[k] = {"A": jload(f"w8_partA_{k}.json")}
        if f"w8_partC_{k}.json" in FILES: RB0[k]["C"] = jload(f"w8_partC_{k}.json"); RB0[k]["P"] = nload(f"w8_perimage_{k}.npz")
        if f"w8_partD_{k}.json" in FILES: RB0[k]["D"] = jload(f"w8_partD_{k}.json")
        gate("RB-0 parts used here are complete", k, all(v.get("complete") for n, v in RB0[k].items() if n != "P"),
             ", ".join(n for n in RB0[k] if n != "P"))
    if f"rb3_{k}.json" in FILES:
        RB3[k] = {"R": jload(f"rb3_{k}.json")}
        if f"rb3_perimage_{k}.npz" in FILES: RB3[k]["P"] = nload(f"rb3_perimage_{k}.npz")
        gate("RB-3 file is complete", k, bool(RB3[k]["R"].get("complete")))
import pandas as pd
RB2MAP = pd.read_csv(FILES["rb2_axis_map.csv"]) if "rb2_axis_map.csv" in FILES else None
print("RB-0:", {k: list(v) for k, v in RB0.items()}, "| RB-3:", list(RB3), "| RB-2 map:", RB2MAP is not None)

In [ ]:
# datasets: explicit ids, no silent fallback
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed, start=0):
            self.n, self.seed, self.start = n, seed, start
        def skip(self, k): return _FakeStream(self.n, self.seed, self.start + k)
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                if i >= self.start: yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1); DS_ACC = _FakeStream(400, 2)
    print("TEST MODE: random images stand in for both datasets")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    DS_ACC = load_imagenet(split="validation", streaming=True, dataset_id=ACC_DATASET, fallback_id=None)
    gate("SSDC dataset is the repack (W2-W6, RB-0 to RB-6)", "data", SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)
    gate("top-1 dataset is the ILSVRC split (W7, RB-0 to RB-6)", "data", ACC_DATASET == "ILSVRC/imagenet-1k", ACC_DATASET)

In [ ]:
# rotary helpers: from W5 / RB-0 / RB-5 / RB-6
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def plane_columns(p, n, rh): return [p, p + n] if rh else [2 * p, 2 * p + 1]

def derive_axis(emb, n_planes, rh, H, W):
    sin, cos = emb.detach().float().chunk(2, -1); rows = []; dec = True
    for p in range(n_planes):
        c = plane_columns(p, n_planes, rh)[0]
        z = torch.complex(cos[:, c], sin[:, c]).reshape(H, W)
        ra = (z - z.mean(dim=1, keepdim=True)).abs().mean().item()
        rb = (z - z.mean(dim=0, keepdim=True)).abs().mean().item()
        ax = "row" if ra < rb else "col"; lo, hi = sorted([ra, rb]); r = lo / hi if hi > 1e-12 else 1.0
        if r > 0.10: dec = False
        rows.append({"plane": p, "axis": ax, "ratio": r})
    return rows, [r["plane"] for r in rows if r["axis"] == "row"], [r["plane"] for r in rows if r["axis"] == "col"], dec

def mask_planes(emb, kill, n, rh):
    if not kill: return emb
    s, c = emb.chunk(2, -1); s, c = s.clone(), c.clone()
    for p in kill:
        for col in plane_columns(p, n, rh): s[..., col] = 0.0; c[..., col] = 1.0
    return torch.cat([s, c], -1)

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

def per_head(heads, inner, n_heads):
    hs = sorted(set(heads))
    def tf(emb):
        if emb.dim() != 2:
            raise RuntimeError(f"per_head expected 2-D rope, got {tuple(emb.shape)}: stacked overrides")
        out = emb.unsqueeze(0).repeat(n_heads, 1, 1)
        if hs:
            t = inner(emb)
            for h in hs: out[h] = t
        return out
    return tf

class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class RopeGlobal:          # W8 / RB-0 / RB-6: a transform of the rotary table in every block (floor = identity)
    def __init__(s, model, tf): s.name, s.mod = find_rope(model); s.tf = tf; s.handles = []
    def __enter__(s):
        if s.mod is None: raise RuntimeError("no get_embed module")
        s.orig = s.mod.get_embed; o, t = s.orig, s.tf
        s.mod.get_embed = (lambda _o=o, _t=t: (lambda *a, **k: _t(_o(*a, **k))))()
        s.handles = [1]; return s
    def __exit__(s, *e): s.mod.get_embed = s.orig; s.handles = []

class APEZero:             # W1b / W8 / RB-6: the whole learned embedding set to zero
    def __init__(s, model): s.model = model; s.handles = []
    def __enter__(s):
        cands = [n for n, _ in s.model.named_parameters() if n.endswith("pos_embed") or n.endswith("position_embeddings")]
        if not cands: raise RuntimeError("no positional-embedding parameter")
        s.name = sorted(cands, key=len)[0]; parts = s.name.split("."); m = s.model
        for p in parts[:-1]: m = getattr(m, p)
        s.parent, s.attr = m, parts[-1]
        pe = getattr(s.parent, s.attr); s.saved = pe.detach().clone()
        with torch.no_grad(): pe.zero_()
        s.handles = [1]; return s
    def __exit__(s, *e):
        with torch.no_grad(): getattr(s.parent, s.attr).copy_(s.saved)
        s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

class Multi:
    def __init__(s, *ctxs): s.ctxs = ctxs; s.handles = []
    def __enter__(s):
        for c in s.ctxs: c.__enter__()
        s.handles = [h for c in s.ctxs for h in (getattr(c, "handles", []) or [])]; return s
    def __exit__(s, *e):
        for c in reversed(s.ctxs): c.__exit__(*e)
        s.handles = []

def win_plus(model, source, heads_by_block, n_heads, window, window_tf=IDEN, inner=IDEN):
    """Rotations off in the window (window_tf), plus `inner` on the listed heads of the listed blocks (as RB-0, with
    the window as an argument)."""
    segs = [(list(window), window_tf)] if window else []
    for k in sorted(heads_by_block):
        if k in window: raise ValueError(f"block {k} lies inside the window")
        hs = sorted(set(heads_by_block[k]))
        if hs: segs.append(([k], per_head(hs, inner, n_heads)))
    return RopeComposite(model, source, segs)

In [ ]:
# algebra gates (as in W5 / RB-0 / RB-5)
def _rot_il(x):
    x = x.unflatten(-1, (-1, 2)); return torch.stack([-x[..., 1], x[..., 0]], -1).flatten(-2)
def _rot_h(x):
    a, b = x.chunk(2, -1); return torch.cat([-b, a], -1)
def _apply(x, e, half=False):
    s, c = e.chunk(2, -1); return x * c + (_rot_h(x) if half else _rot_il(x)) * s

for lname, rh in [("interleaved", False), ("rotate_half", True)]:
    npl, dim, H_ = 32, 64, 12
    e = torch.randn(196, 2 * dim); x = torch.randn(2, H_, 196, dim)
    gate(f"empty plane mask bit-identical [{lname}]", "algebra", torch.equal(mask_planes(e, [], npl, rh), e))
    gate(f"full plane mask equals identity [{lname}]", "algebra",
         torch.allclose(mask_planes(e, list(range(npl)), npl, rh), identity_like(e)))
    m = mask_planes(e, [3], npl, rh)
    touched = torch.nonzero((m - e).abs().sum(0) > 0).flatten().tolist()
    sc = plane_columns(3, npl, rh); exp = sorted(set(sc + [c + dim for c in sc]))
    gate(f"plane 3 touches only its own columns [{lname}]", "algebra", touched == exp, f"{touched} vs {exp}")
    full = _apply(x, e, rh)
    gate(f"mask NO heads equals unmodified rope [{lname}]", "per-head",
         torch.allclose(_apply(x, per_head([], identity_like, H_)(e), rh), full, atol=1e-6))
    one = _apply(x, per_head([3], identity_like, H_)(e), rh); others = [h for h in range(H_) if h != 3]
    gate(f"mask head 3 leaves others untouched [{lname}]", "per-head", torch.allclose(one[:, others], full[:, others], atol=1e-6))
    gate(f"mask head 3 changes head 3 [{lname}]", "per-head", not torch.allclose(one[:, 3], full[:, 3]))

class _Stub:
    def __init__(s, n=12): s.blocks = [nn.Identity() for _ in range(n)]
try:
    _wp = win_plus(_Stub(), "timm", {7: [3]}, 12, window=list(range(6)))
    dj = [b for b, _ in _wp.segs] == [list(range(6)), [7]]
except Exception:
    dj = False
gate("win_plus builds the window plus one segment for the single head's block", "composite", dj)
try:
    win_plus(_Stub(), "timm", {5: [0]}, 12, window=list(range(6))); bad = False
except ValueError:
    bad = True
gate("win_plus REJECTS a single head inside the window", "composite", bad)

In [ ]:
# image cache: decode once per model, replay through a real DataLoader (RB-0)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(im.clone(), torch.as_tensor(lb).clone()) for im, lb in dl]
    print(f"    cached {sum(int(lb.numel()) for _, lb in out)} images in {time.time() - t:.0f}s", flush=True)
    return out

def cache_digest(cache):
    h = hashlib.sha256()
    for im, lb in cache[:4]: h.update(im.numpy().tobytes()); h.update(lb.numpy().tobytes())
    return h.hexdigest()[:16]

In [ ]:
# measurement: RB-0 / RB-2's measure(), for any number of blocks
def final_norm_module(model):
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

def measure(M, ctx, capture=()):
    """One RPI sweep over the cached repack images under `ctx`: SSDC at every block input and at FINAL (= n_blocks),
    plus captured patch tokens."""
    model, source, n_prefix, NB = M["model"], M["source"], M["n_prefix"], M["n_blocks"]
    cache = M["cache"]["ssdc"]; FIN = NB
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    handles = _register_ssdc_accumulator_hooks(model, source, acc)
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    cap = {r: [] for r in capture}
    def keep(r, tok):
        if r in cap: cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())
    extra = [model.register_forward_pre_hook(
        lambda m, args: st.update(cur=st["next"], next=st["next"] + int(args[0].shape[0])))]
    for i in range(NB):
        if i in cap:
            extra.append(get_block_attention(blocks[i], source).register_forward_hook(
                lambda m, inp, out, i=i: keep(i, inp[0].detach())))
    def final_hook(m, inp, out):
        t = fnorm(out.detach()); acc.add(FIN, t); keep(FIN, t)
    extra.append(blocks[-1].register_forward_hook(final_hook))
    t0 = time.time()
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles + extra: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(NB)] + [acc.ssdc(FIN, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"]),
           "runtime_s": round(time.time() - t0, 1)}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

# probes: W5's per-block-seeded protocol with RB-2's noise recipe
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0; best = None
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0; preds = []
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1); preds.append(p)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        acc_ = correct / Xte.shape[0]
        if acc_ > peak: peak = acc_; best = torch.cat(preds)
    del head, opt
    return float(peak), best

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block, heads=("exact", "row", "col"), noise=False):
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, te_img = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name in heads:
        lab, nc = labels[name]; y = lab.repeat(n_images)
        acc_, preds = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
        out[name] = acc_
        if noise:
            corr = (preds == y[te].to(PDEV)).float()
            g = np.random.default_rng(TORCH_SEED + block)
            n_te = len(te_img); per = corr.reshape(n_te, T).mean(1).cpu().numpy()
            bt = [per[g.integers(0, n_te, n_te)].mean() for _ in range(NOISE_BOOT)]
            seeds = [_fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block + 1000 * s)[0]
                     for s in NOISE_SEEDS]
            out.setdefault("noise", {})[name] = {
                "boot_sd": float(np.std(bt, ddof=1)), "seed_sd": float(np.std(seeds, ddof=1)),
                "seed_values": [float(v) for v in seeds],
                "combined_sd": float(np.sqrt(np.var(bt, ddof=1) + np.var(seeds, ddof=1)))}
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

def accuracy(M, ctx):
    model, cache = M["model"], M["cache"]["acc"]
    model.eval(); correct = []
    torch.manual_seed(TORCH_SEED)
    with ctx:
        hooks = len(getattr(ctx, "handles", []) or [])
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                pred = model(images.to(DEVICE)).argmax(-1).cpu()
                correct.append(pred == torch.as_tensor(labels).cpu())
    c = torch.cat(correct)
    return float(c.float().mean()), c.numpy().astype(np.uint8), hooks

In [ ]:
# timm model setup, shared by parts d, a, b, e
TEST_INIT_SEED = {"naver": 101, "DINOv3": 102, "small": 103, "reg1_gap": 104}    # rope_ape: 0 (as the test W8)
def load_timm(key, res):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        tiny = dict(embed_dim=96, num_heads=6 if key != "naver" else 12, pretrained=False, dynamic_img_size=True,
                    num_classes=10 if spec["classifier"] else 0)
        if key == "naver": tiny["embed_dim"] = 192
        torch.manual_seed(TEST_INIT_SEED.get(key, 0) if key != "large" else 105)
        model = timm.create_model(spec["name"].split(".")[0], **tiny).to(DEVICE).eval()
        with torch.no_grad():
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
                for pn in ("q_proj", "k_proj"):
                    m_ = getattr(blk.attn, pn, None)
                    if m_ is not None: m_.weight.mul_(4.0)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, res, res)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=res)

def first_layer_rel_diff(M):
    model, source = M["model"], M["source"]
    conv = get_patch_embed_conv(model, source); grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    try:
        with torch.no_grad(): model(M["cache"]["ssdc"][0][0].to(DEVICE))
    finally:
        h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    n = int(grab["x"].shape[0]); del grab, y64
    return v, n

def setup_timm(key, scope, res=224, need_acc=False, n_ssdc=None, replay=True):
    """Load, cache, and gate one timm RoPE model. `scope` labels the gates (e.g. 'd:naver')."""
    spec = MODEL_SPECS[key]; n_ssdc = n_ssdc or N_SSDC
    print("\n" + "=" * 78 + f"\n[{scope}] {spec['name']} at {res}x{res}\n" + "=" * 78, flush=True)
    model, processor, source = load_timm(key, res)
    model.eval().float()
    rh, rh_src, nh = attn_attrs(model, source)
    blocks = get_vit_blocks(model, source)
    M = {"key": key, "scope": scope, "res": res, "model": model, "source": source, "processor": processor,
         "n_prefix": int(getattr(model, "num_prefix_tokens", 1)), "rotate_half": bool(rh), "n_heads": int(nh),
         "n_blocks": len(blocks), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, n_ssdc)
    if need_acc and spec["classifier"]:
        M["cache"]["acc"] = build_cache(DS_ACC, processor, source, N_ACC)
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(M["cache"]["ssdc"][0][0][:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg); T = Hg * Wg
    gate(f"forward grid is {res // 16}x{res // 16}", scope, (Hg, Wg) == (res // 16, res // 16), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    rname, rmod = find_rope(model); M["rope"] = rmod
    emb0 = rmod.get_embed(shape=(Hg, Wg)); n_planes = int(emb0.shape[-1] // 4); M["n_planes"] = n_planes
    amap, rowp, colp, dec = derive_axis(emb0, n_planes, rh, Hg, Wg)
    M["row_planes"], M["col_planes"] = rowp, colp
    gate("rope rows equal patch tokens; no head dimension", scope, int(emb0.shape[0]) == T and emb0.dim() == 2,
         f"shape {tuple(emb0.shape)}; {n_planes} planes per head")
    gate("every plane is a row plane or a column plane (decisive), half each", scope,
         dec and len(rowp) == len(colp) == n_planes // 2, f"{len(rowp)} row / {len(colp)} col; max ratio {max(a['ratio'] for a in amap):.1e}")
    M["ROWM"] = lambda e, _r=rowp, _n=n_planes, _h=rh: mask_planes(e, _r, _n, _h)
    M["COLM"] = lambda e, _c=colp, _n=n_planes, _h=rh: mask_planes(e, _c, _n, _h)
    rp = sorted(np.random.default_rng(RAND_PLANES_SEED).choice(n_planes, size=n_planes // 2, replace=False).tolist())
    M["rand_planes"] = rp
    M["RANDM"] = lambda e, _p=rp, _n=n_planes, _h=rh: mask_planes(e, _p, _n, _h)
    ra = (RB0.get(key) or {}).get("A")
    if ra: gate("num_heads matches RB-0", scope, M["n_heads"] == ra["n_heads"], f"{M['n_heads']} vs {ra['n_heads']}")
    print(f"    heads {nh} | blocks {len(blocks)} | prefix {M['n_prefix']} | rotate_half {rh} | planes {n_planes} | final norm {fn_name}")
    if DEVICE == "cuda" and not TEST_MODE:
        v, nb = first_layer_rel_diff(M); M["first_layer_rel_diff"] = v
        if key in FIRST_LAYER_910 and res == 224:
            exp_ = FIRST_LAYER_910[key]
            gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value (TF32-level kernel, as in September)", scope,
                 nb == BATCH_SIZE and abs(v / exp_ - 1) <= 0.01, f"{v:.4e} vs {exp_:.4e} (batch {nb})")
        else:   # no RB-1b value for this model / input size: record it (the pre-flight fixes the environment)
            gate("patch embedding agrees with a float64 computation to TF32 level or better (recorded)", scope,
                 nb == BATCH_SIZE and np.isfinite(v) and v <= FIRST_LAYER_RANGE[1], f"{v:.4e} (batch {nb})")
    else:
        gate("first-layer numerics", scope, True, "test mode / CPU", skip=True)
    if replay:
        def streamed_ssdc(ctx):
            torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
            with ctx:
                s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=n_ssdc,
                                     batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
            return s
        a = streamed_ssdc(NullCtx()); b = measure(M, NullCtx())["ssdc"][:M["n_blocks"]]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate("cached replay reproduces streamed evaluate_ssdc [intact]", scope, d <= EXACT_TOL, f"max |delta| {d:.2e}")
        if "acc" in M["cache"]:
            n_chk = min(REPLAY_CHECK_N, N_ACC)
            dl = prep_data(DS_ACC, processor, source, corruption_type=None, number_images=n_chk,
                           batch_size=BATCH_SIZE, half=False, num_workers=0)
            cs = []
            with torch.inference_mode():
                for im, lb in dl: cs.append(model(im.to(DEVICE)).argmax(-1).cpu() == torch.as_tensor(lb))
            cs = torch.cat(cs).numpy().astype(np.uint8)
            _, cc, _ = accuracy(M, NullCtx())
            gate(f"cached replay reproduces streamed top-1 on the first {n_chk} images", scope,
                 np.array_equal(cs, cc[:n_chk]), f"{int((cs != cc[:n_chk]).sum())} images differ")
    M["cache_digest"] = cache_digest(M["cache"]["ssdc"])
    return M

def attention_level_gate(M, window, block, head, scope):
    """Masking one (block, head) after the window changes that head's attention output and no other (RB-2)."""
    model, source, nh = M["model"], M["source"], M["n_heads"]
    grab = {}
    proj = get_block_attention(get_vit_blocks(model, source)[block], source).proj
    hk = proj.register_forward_pre_hook(lambda m, args: grab.__setitem__("x", args[0].detach().clone()))
    xb = M["cache"]["ssdc"][0][0][:4].to(DEVICE)
    try:
        with torch.no_grad():
            with RopeComposite(model, source, [(window, IDEN)]): model(xb)
            ref_x = grab["x"]
            with win_plus(model, source, {block: [head]}, nh, window): model(xb)
            new_x = grab["x"]
    finally:
        hk.remove()
    hd = ref_x.shape[-1] // nh
    same = all(torch.equal(ref_x[..., h * hd:(h + 1) * hd], new_x[..., h * hd:(h + 1) * hd]) for h in range(nh) if h != head)
    changed = not torch.equal(ref_x[..., head * hd:(head + 1) * hd], new_x[..., head * hd:(head + 1) * hd])
    gate(f"masking (block {block}, head {head}) after the window changes that head's attention output and leaves the other "
         f"{nh - 1} bit-identical", scope, same and changed)

def release(M):
    for k in list(M.keys()): M[k] = None
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

# generic bookkeeping: one JSON + one per-image npz per (part, model)
CFG_BASE = {"n_ssdc": N_SSDC, "n_acc": N_ACC, "batch": BATCH_SIZE, "seed": TORCH_SEED, "ssdc_dataset": SSDC_DATASET,
            "acc_dataset": ACC_DATASET, "metric": METRIC, "probe": [PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC],
            "noise": [NOISE_SEEDS, NOISE_BOOT], "test_mode": TEST_MODE}
SESSION = {"session": RUN_STAMP, "torch": torch.__version__, "cudnn": ENV["cudnn_reported"], "python": sys.executable,
           "gpu": ENV["gpu"], "repo_commit": REPO_COMMIT}
def rfile(part, key): return OUT_DIR / f"rb7{part}_{key}.json"
def pfile(part, key): return OUT_DIR / f"rb7{part}_perimage_{key}.npz"
def load_R(part, key, cfg):
    f = rfile(part, key)
    if not (RESUME and f.exists()): return None, {}
    R = json.loads(f.read_text())
    if R.get("config") != cfg:
        raise SystemExit(f"{f.name} was written with a different configuration; move it away or set RB7_RESUME=0.")
    P = {n: z_[n] for z_ in [np.load(pfile(part, key))] for n in z_.files} if pfile(part, key).exists() else {}
    for n in R.get("acc", {}):
        if n not in P: raise SystemExit(f"{pfile(part, key).name} lacks per-image results for {n}; set RB7_RESUME=0.")
    return R, P
def save_R(part, key, R, P):
    R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    if P:
        t = pfile(part, key).with_name(pfile(part, key).stem + ".tmp.npz"); np.savez_compressed(t, **P); os.replace(t, pfile(part, key))
    t = rfile(part, key).with_suffix(".json.tmp"); t.write_text(json.dumps(R, indent=1, default=float)); os.replace(t, rfile(part, key))
def _maxdiff(a, b): return max((abs(x - y) for x, y in zip(a, b)), default=0.0)
def session_gate(R, scope):
    if TEST_MODE: return
    ses = R.get("sessions", [])
    gate("every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002", scope,
         bool(ses) and all(x.get("cudnn") == EXPECT_CUDNN and x.get("torch") == EXPECT_TORCH for x in ses), f"{len(ses)} session(s)")

## Part d - is the six-head result a block-position effect?
Section 4.5 (M20) says the six top SSDC heads together cost more top-1 than each of 24 random sets of one head per
block (RB-3). Those random sets put half their heads in blocks 9-11, where single heads cost little, while the top
six sit in blocks 6-8. Here the null keeps the top six's block counts exactly: 48 random sets per
model, heads drawn uniformly within each block. The top-6 set, two of RB-0's one-per-block sets, the window and Cref
are re-measured first and must equal RB-3 / RB-0 bit for bit.

In [ ]:
def rank_ssdc_b11(RA):
    """RB-2's pre-registered rule (as RB-3/RB-5): heads in blocks 6-10 ranked by SSDC lost entering block 11 vs Cref."""
    c = RA["conditions"]; nh = RA["n_heads"]; cref = c["Cref"]["ssdc"]
    r = sorted([(cref[11] - c[f"s_b{k}_h{h}"]["ssdc"][11], k, h) for k in D_POST[:-1] for h in range(nh)],
               key=lambda t: (-t[0], t[1], t[2]))
    return [(k, h) for _, k, h in r]

def draw_block_matched(top6, nh, n, seed):
    """n distinct random sets with exactly top6's block counts; heads uniform without replacement within a block."""
    g = np.random.default_rng(seed); target = {int(k): len(v) for k, v in top6.items()}
    ref = {int(k): sorted(v) for k, v in top6.items()}; out, seen, tries = [], set(), 0
    while len(out) < n:
        tries += 1
        if tries > 100000: raise RuntimeError("could not draw enough distinct block-matched sets")
        s = {k: sorted(g.choice(nh, size=c, replace=False).tolist()) for k, c in sorted(target.items())}
        key_ = json.dumps(s, sort_keys=True)
        if s == ref or key_ in seen: continue
        seen.add(key_); out.append(s)
    return out

def set_hooks(sel): return len(D_WINDOW) + sum(1 for v in sel.values() if v)

def run_part_d():
    for i_m, key in enumerate(D_MODELS):
        try:
            run_d_model(i_m, key)
        except SystemExit:
            raise
        except Exception as ex:
            traceback.print_exc(); gate("part d model ran to the end", f"d:{key}", False, repr(ex)[:300])

def run_d_model(i_m, key):
    if True:
        scope = f"d:{key}"
        cls = MODEL_SPECS[key]["classifier"]
        R3 = RB3[key]["R"]; top6 = {int(k): v for k, v in R3["selection"]["top6"].items()}
        r610 = rank_ssdc_b11(RB0[key]["A"]); mine = {}
        for k, h in r610[:6]: mine.setdefault(k, []).append(h)
        gate("RB-3's top-6 set equals RB-2's rule recomputed from RB-0 part A", scope,
             {k: sorted(v) for k, v in mine.items()} == {k: sorted(v) for k, v in top6.items()}, f"{top6}")
        nh = RB0[key]["A"]["n_heads"]
        sets = draw_block_matched(top6, nh, D_N_NULL, D_SEED + i_m)
        cfg = {**CFG_BASE, "part": "d", "window": D_WINDOW, "top6": {str(k): v for k, v in top6.items()},
               "null_seed": D_SEED + i_m, "n_null": D_N_NULL}
        R, P = load_R("d", key, cfg)
        if R is None:
            R, P = {"model": key, "part": "d", "config": cfg, "n_heads": nh, "sets": sets, "ssdc": {}, "acc": {}, "sessions": []}, {}
        gate("block-matched sets are the pre-registered draw (same seed, same sets)", scope,
             json.loads(json.dumps(R["sets"])) == json.loads(json.dumps(sets)), f"{len(sets)} sets")
        rand0 = RB0[key]["D"]["random_sets"]
        arms = {"Cref": {}, "top6": top6}
        for j in range(D_N_RAND_GATE): arms[f"rand{j:02d}"] = {k: [h] for k, h in zip(D_POST, rand0[j])}
        for j, s in enumerate(sets): arms[f"null{j:02d}"] = s
        need_acc = cls and any(n not in R["acc"] for n in ["window"] + list(arms)[1:])
        need_ssdc = any(n not in R["ssdc"] for n in arms)
        if not (need_acc or need_ssdc):
            print(f"\n[{scope}] already complete; re-checking"); RESULTS.setdefault("d", {})[key] = R; check_part_d(key, R, P); return
        try:
            M = setup_timm(key, scope, 224, need_acc=cls)
        except Exception as ex:
            traceback.print_exc(); gate("model setup completed", scope, False, repr(ex)[:200]); return
        if scope_failed(scope) and STRICT and not TEST_MODE:
            gate("part d measured", scope, False, "STOPPED: a setup gate failed (RB7_STRICT=1)"); release(M); return
        R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff")})
        model, source = M["model"], M["source"]; t0 = time.time()
        def ctx_of(name):
            if name in ("Cref", "window"): return RopeComposite(model, source, [(D_WINDOW, IDEN)])
            return win_plus(model, source, arms[name], nh, D_WINDOW)
        order = ["Cref", "top6"] + [f"rand{j:02d}" for j in range(D_N_RAND_GATE)] + [f"null{j:02d}" for j in range(len(sets))]
        for i, name in enumerate(order):
            if name not in R["ssdc"]:
                r = measure(M, ctx_of(name)); R["ssdc"][name] = {"ssdc": r["ssdc"], "hooks": r["hooks"], "n_images": r["n_images"]}
            an = "window" if name == "Cref" else name
            if cls and an not in R["acc"]:
                a, c, hk = accuracy(M, ctx_of(an)); R["acc"][an] = {"acc": a, "hooks": hk, "n_images": int(len(c))}; P[an] = c
            if i == 1 + D_N_RAND_GATE:     # references and gates measured: check before the null
                save_R("d", key, R, P)
                if not check_part_d(key, R, P, refs_only=True) and STRICT and not TEST_MODE:
                    gate("part d measured", scope, False, "STOPPED: a reference gate failed (RB7_STRICT=1)"); break
            if i % 8 == 7 or i == len(order) - 1:
                save_R("d", key, R, P); print(f"    {i + 1}/{len(order)} arms | {(time.time() - t0) / 60:.1f} min", flush=True)
        R["complete"] = all(n in R["ssdc"] for n in order) and (not cls or all(("window" if n == "Cref" else n) in R["acc"] for n in order))
        save_R("d", key, R, P); check_part_d(key, R, P)
        RESULTS.setdefault("d", {})[key] = R
        release(M)

def check_part_d(key, R, P, refs_only=False):
    scope = f"d:{key}"; ok = []; cls = MODEL_SPECS[key]["classifier"]
    A = RB0[key]["A"]["conditions"]; D = RB0[key]["D"]; R3 = RB3[key]["R"]
    s = R["ssdc"]
    ok.append(gate("Cref SSDC equals RB-0 part A bit for bit (13 readouts)", scope, s["Cref"]["ssdc"] == A["Cref"]["ssdc"]))
    ok.append(gate("top-6 SSDC equals RB-3 bit for bit", scope, s["top6"]["ssdc"] == R3["ssdc"]["ssdc_top6"]["ssdc"]))
    for j in range(D_N_RAND_GATE):
        ok.append(gate(f"RB-0 part D one-per-block set {j} SSDC reproduces bit for bit", scope,
                       s[f"rand{j:02d}"]["ssdc"] == D["ssdc"][f"rand{j:02d}"]))
    if cls:
        C = RB0[key]["C"]["conditions"]; Pc = RB0[key]["P"]; P3 = RB3[key]["P"]
        ok.append(gate("window top-1 equals RB-0 part C bit for bit (mean and per image)", scope,
                       R["acc"]["window"]["acc"] == C["window"]["acc"] and np.array_equal(P["window"], Pc["window"])))
        ok.append(gate("top-6 top-1 equals RB-3 bit for bit (mean and per image)", scope,
                       R["acc"]["top6"]["acc"] == R3["acc"]["top6"]["acc"] and np.array_equal(P["top6"], P3["top6"])))
        for j in range(D_N_RAND_GATE):
            ok.append(gate(f"RB-0 part D one-per-block set {j} top-1 reproduces bit for bit", scope,
                           R["acc"][f"rand{j:02d}"]["acc"] == D["acc"][f"rand{j:02d}"]))
    if refs_only: return all(ok)
    names = [n for n in s if n.startswith("null")]
    gate("every null set measured (SSDC" + (" and top-1)" if cls else ")"), scope,
         len(names) == len(R["sets"]) and (not cls or all(n in R["acc"] for n in names)), f"{len(names)}/{len(R['sets'])}")
    badh = [n for n, sel in [("top6", R["config"]["top6"])] + [(f"null{j:02d}", st) for j, st in enumerate(R["sets"])]
            if n in s and s[n]["hooks"] != set_hooks(sel)]
    badh += [n for n in ("Cref",) if s[n]["hooks"] != len(D_WINDOW)]
    gate("every arm registered the expected hooks (window + one per block with heads)", scope, not badh, ",".join(badh[:5]))
    badn = [n for n in s if s[n]["n_images"] != N_SSDC] + [n for n in R["acc"] if R["acc"][n]["n_images"] != N_ACC]
    gate("every arm saw every image", scope, not badn, ",".join(badn[:5]))
    blk = [n for n in names if s[n]["ssdc"][:6] != s["Cref"]["ssdc"][:6]]
    gate("null sets leave readouts 0-6 bit-identical to Cref (heads act from block 6 on)", scope, not blk, ",".join(blk[:5]))
    session_gate(R, scope)
    return all(ok)

## Parts a and b - the paper's RoPE tests at 320×320 (a) and in ViT-L (b)
One runner for both. References (with probes): intact, floor, the first-half window (`Cref`), the same window with
only the row or only the column planes off (`Dref_row`, `Dref_col`, with probe noise), and one axis's planes (or
16 random planes) off in every block. Then every single head after the window, rotations off while the rest
recover: SSDC (1,000 repack images) and top-1 (5,000 ILSVRC images). Then the axis map (RB-2's windows and noise
bar) for the heads selected by the fixed rule. Part a's intact, floor and Cref SSDC must equal W6's 320×320 run bit
for bit; ViT-L has no earlier run.

In [ ]:
def scale_spec(part, key):
    if part == "a":
        return {"res": A_RES, "window": list(range(0, 6)), "post": list(range(6, 12)), "axis_r": 4, "extra": {}}
    return {"res": 224, "window": B_WINDOWS[B_MAIN], "post": list(range(B_WINDOWS[B_MAIN][-1] + 1, 24)), "axis_r": 8,
            "extra": {k: v for k, v in B_WINDOWS.items() if k != B_MAIN}}

def single_name(k, h): return f"s_b{k}_h{h}"

def ssdc_rank(R, sp):
    """RB-2's rule: heads in blocks before the last ranked by SSDC lost entering the last block vs Cref."""
    last = sp["post"][-1]; cref = R["ssdc"]["Cref"]["ssdc"]; nh = R["n_heads"]
    r = sorted([(cref[last] - R["ssdc"][single_name(k, h)]["ssdc"][last], k, h) for k in sp["post"][:-1] for h in range(nh)
                if single_name(k, h) in R["ssdc"]], key=lambda t: (-t[0], t[1], t[2]))
    return [(k, h, v) for v, k, h in r]

def top1_rank(R, P, sp):
    nh = R["n_heads"]; win = R["acc"]["window"]["acc"]
    r = sorted([((win - R["acc"][single_name(k, h)]["acc"]) * 100, k, h) for k in sp["post"] for h in range(nh)
                if single_name(k, h) in R["acc"]], key=lambda t: (-t[0], t[1], t[2]))
    return [(k, h, v) for v, k, h in r]

def top1_224(key):
    C = RB0[key]["C"]["conditions"]; win = C["window"]["acc"]; nh = RB0[key]["A"]["n_heads"]
    return {(k, h): (win - C[single_name(k, h)]["acc"]) * 100 for k in range(6, 12) for h in range(nh) if single_name(k, h) in C}

def ref_head_of(key, nh):
    k, h = A_REF_HEAD[key]
    return (k, h % nh) if TEST_MODE else (k, h)      # test mode: tiny models have fewer heads

def select_axis_heads(part, key, R, P, sp):
    sel = set((k, h) for k, h, _ in ssdc_rank(R, sp)[:N_AXIS_TOP])
    if MODEL_SPECS[key]["classifier"]: sel |= set((k, h) for k, h, _ in top1_rank(R, P, sp)[:N_AXIS_TOP])
    if part == "a":
        t = sorted(top1_224(key).items(), key=lambda kv: (-kv[1], kv[0]))[:2]
        sel |= set(kh for kh, _ in t); sel.add(ref_head_of(key, R["n_heads"]))
    return sorted(sel)

def run_scale(part, key):
    sp = scale_spec(part, key); scope = f"{part}:{key}"; cls = MODEL_SPECS[key]["classifier"]
    window, post, axis_r = sp["window"], sp["post"], sp["axis_r"]
    cfg = {**CFG_BASE, "part": part, "res": sp["res"], "window": window, "post": post, "axis_readout": axis_r,
           "extra_windows": sp["extra"], "n_axis_top": N_AXIS_TOP, "rand_planes_seed": RAND_PLANES_SEED}
    R, P = load_R(part, key, cfg)
    if R is not None and R.get("complete"):
        print(f"\n[{scope}] already complete; re-checking"); check_scale(part, key, R, P); return R
    try:
        M = setup_timm(key, scope, sp["res"], need_acc=cls)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", scope, False, repr(ex)[:200]); return None
    if scope_failed(scope) and STRICT and not TEST_MODE:
        gate(f"part {part} measured", scope, False, "STOPPED: a setup gate failed (RB7_STRICT=1)"); release(M); return None
    model, source, nh, NB = M["model"], M["source"], M["n_heads"], M["n_blocks"]
    FIN, last = NB, NB - 1; H, W = M["grid"]
    gate("window, post-window blocks and readouts fit the model", scope,
         post[-1] == last and window[0] == 0 and window[-1] + 1 == post[0], f"{NB} blocks; window {window[0]}-{window[-1]}")
    if R is None:
        R, P = {"model": key, "name": MODEL_SPECS[key]["name"], "part": part, "config": cfg, "n_heads": nh, "n_blocks": NB,
                "grid": [H, W], "row_planes": M["row_planes"], "col_planes": M["col_planes"], "rand_planes": M["rand_planes"],
                "readouts": f"0-{last} = input of block b after its first LayerNorm; {FIN} = output of block {last} through "
                            f"model.{M['final_norm_name']}", "ssdc": {}, "acc": {}, "sessions": []}, {}
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff"), "cache_digest": M.get("cache_digest")})
    if part == "a":
        _, r224, c224, _ = derive_axis(M["rope"].get_embed(shape=(14, 14)), M["n_planes"], M["rotate_half"], 14, 14)
        gate("rotation planes on the 20x20 grid are the same row/column planes as on the 14x14 grid", scope,
             r224 == M["row_planes"] and c224 == M["col_planes"], f"row {M['row_planes'][:4]}...")
    TF = {"cref": IDEN, "row": M["ROWM"], "col": M["COLM"]}
    t0 = time.time()
    def run_ssdc(name, ctx, readouts=(), noise_at=(), heads=("exact", "row", "col")):
        if name in R["ssdc"]: return
        r = measure(M, ctx, capture=tuple(readouts))
        pr = {str(b): probe_block(r["acts"][b], H, W, b, heads=heads, noise=(b in noise_at)) for b in readouts}
        R["ssdc"][name] = {"ssdc": r["ssdc"], "hooks": r["hooks"], "n_images": r["n_images"], "probes": pr}
        del r
    def run_acc(name, ctx):
        if not cls or name in R["acc"]: return
        a, c, hk = accuracy(M, ctx); R["acc"][name] = {"acc": a, "hooks": hk, "n_images": int(len(c))}; P[name] = c
    print(f"\n--- references: {scope} ---", flush=True)
    run_ssdc("baseline", NullCtx(), [axis_r, last, FIN])
    run_ssdc("floor", RopeGlobal(model, IDEN), [axis_r, last, FIN])
    run_ssdc("Cref", RopeComposite(model, source, [(window, IDEN)]), [last, FIN], (last, FIN), ("row", "col"))
    run_ssdc("Dref_row", RopeComposite(model, source, [(window, M["ROWM"])]), [last, FIN], (last, FIN), ("row", "col"))
    run_ssdc("Dref_col", RopeComposite(model, source, [(window, M["COLM"])]), [last, FIN], (last, FIN), ("row", "col"))
    run_ssdc("rowoff_all", RopeGlobal(model, M["ROWM"]), [axis_r])
    run_ssdc("coloff_all", RopeGlobal(model, M["COLM"]), [axis_r])
    run_ssdc("randoff_all", RopeGlobal(model, M["RANDM"]), [axis_r])
    for wn, wb in sp["extra"].items(): run_ssdc(wn, RopeComposite(model, source, [(wb, IDEN)]))
    run_acc("baseline", NullCtx()); run_acc("floor", RopeGlobal(model, IDEN))
    run_acc("window", RopeComposite(model, source, [(window, IDEN)]))
    save_R(part, key, R, P)
    print(f"    references done in {(time.time() - t0) / 60:.1f} min", flush=True)
    if not check_scale(part, key, R, P, refs_only=True) and STRICT and not TEST_MODE:
        gate(f"part {part} measured", scope, False, "STOPPED: a reference gate failed (RB7_STRICT=1)"); release(M); return R
    ref_head = ref_head_of(key, nh) if part == "a" else (post[0], 0)
    attention_level_gate(M, window, ref_head[0], ref_head[1], scope)
    todo = [(k, h) for k in post for h in range(nh) if single_name(k, h) not in R["ssdc"] or (cls and single_name(k, h) not in R["acc"])]
    print(f"\n--- single heads: {scope}, {len(todo)} to measure ---", flush=True); t1 = time.time()
    for i, (k, h) in enumerate(todo):
        ctx = lambda: win_plus(model, source, {k: [h]}, nh, window)
        run_ssdc(single_name(k, h), ctx()); run_acc(single_name(k, h), ctx())
        if h == nh - 1 or i == len(todo) - 1:
            save_R(part, key, R, P); el = time.time() - t1
            print(f"    block {k} done | {i + 1}/{len(todo)} | {el / 60:.1f} min, ~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min left", flush=True)
    sel = select_axis_heads(part, key, R, P, sp); R["axis_heads"] = [list(x) for x in sel]
    print(f"\n--- axis map: {scope}, heads {sel} ---", flush=True)
    for k, h in sel:
        ro = [last] if k < last else [FIN]
        for w in ("row", "col"):
            run_ssdc(f"{w}_b{k}_h{h}", win_plus(model, source, {k: [h]}, nh, window, window_tf=TF[w]), ro, (), ("row", "col"))
        save_R(part, key, R, P)
    R["complete"] = (all(single_name(k, h) in R["ssdc"] for k in post for h in range(nh))
                     and (not cls or all(single_name(k, h) in R["acc"] for k in post for h in range(nh)))
                     and all(f"{w}_b{k}_h{h}" in R["ssdc"] for k, h in sel for w in ("row", "col")))
    R["runtime_min_this_session"] = round((time.time() - t0) / 60, 1)
    save_R(part, key, R, P); check_scale(part, key, R, P)
    gate(f"part {part} finished", scope, True, f"{R['runtime_min_this_session']} min; complete {R['complete']}")
    release(M)
    return R

def expected_hooks_scale(name, sp):
    if name == "baseline": return 0
    if name in ("floor", "rowoff_all", "randoff_all", "coloff_all"): return 1
    if name in ("Cref", "Dref_row", "Dref_col", "window"): return len(sp["window"])
    if name in sp["extra"]: return len(sp["extra"][name])
    return len(sp["window"]) + 1

def check_scale(part, key, R, P, refs_only=False):
    sp = scale_spec(part, key); scope = f"{part}:{key}"; ok = []; s = R["ssdc"]; cls = MODEL_SPECS[key]["classifier"]
    NB = R["n_blocks"]; FIN, last = NB, NB - 1
    refs = ["baseline", "floor", "Cref", "Dref_row", "Dref_col", "rowoff_all", "coloff_all", "randoff_all"] + list(sp["extra"])
    miss = [n for n in refs if n not in s] + [n for n in (["baseline", "floor", "window"] if cls else []) if n not in R["acc"]]
    ok.append(gate("every reference measured", scope, not miss, ",".join(miss)))
    if miss: return False
    badh = [n for n in refs if s[n]["hooks"] != expected_hooks_scale(n, sp)] + \
           [n for n in R["acc"] if R["acc"][n]["hooks"] != expected_hooks_scale(n, sp)]
    ok.append(gate("every reference registered the expected hooks", scope, not badh, ",".join(badh)))
    badn = [n for n in s if s[n]["n_images"] != N_SSDC] + [n for n in R["acc"] if R["acc"][n]["n_images"] != N_ACC]
    ok.append(gate("every arm saw every image", scope, not badn, ",".join(badn[:5])))
    _d = f"{s['baseline']['ssdc'][last]:.4f} -> {s['floor']['ssdc'][last]:.4f}"
    if TEST_MODE: gate("floor lowers SSDC entering the last block", scope, True, "test mode: tiny random models have no spatial structure; " + _d, skip=True)
    else: ok.append(gate("floor lowers SSDC entering the last block", scope, s["floor"]["ssdc"][last] < s["baseline"]["ssdc"][last], _d))
    w = sp["window"]; pre = max(_maxdiff(s[n]["ssdc"][:w[0] + 1], s["baseline"]["ssdc"][:w[0] + 1]) for n in ("Cref", "Dref_row", "Dref_col"))
    ok.append(gate("windows leave readout 0 bit-identical to intact (masks act inside attention)", scope, pre <= EXACT_TOL, f"{pre:.1e}"))
    nz = [s[n]["probes"][str(r)].get("noise", {}).get(ax, {}).get("combined_sd") for n in ("Cref", "Dref_row", "Dref_col")
          for r in (last, FIN) for ax in ("row", "col")]
    ok.append(gate("probe noise measured on Cref, Dref_row, Dref_col (entering the last block and FINAL) and finite", scope,
                   all(v is not None and np.isfinite(v) for v in nz), f"{len(nz)} values"))
    if part == "a" and not TEST_MODE:
        w6 = json.loads(FILES[W6_PARTA[key]].read_text())["by_res"][str(A_RES)]["conditions"]
        d = max(_maxdiff(s[n]["ssdc"][:12], w6[n]["ssdc"]) for n in ("baseline", "floor", "Cref"))
        ok.append(gate("320x320 intact, floor and Cref SSDC equal W6 bit for bit (12 readouts each)", scope, d <= EXACT_TOL, f"max |delta| {d:.1e}"))
    elif part == "a":
        gate("320x320 references equal W6", scope, True, "test mode: no W6 anchor for tiny models", skip=True)
    if refs_only: return all(ok)
    nh = R["n_heads"]; names = [(k, h) for k in sp["post"] for h in range(nh)]
    m_s = [single_name(k, h) for k, h in names if single_name(k, h) not in s]
    m_a = [single_name(k, h) for k, h in names if cls and single_name(k, h) not in R["acc"]]
    gate("every single head measured (SSDC" + (" and top-1)" if cls else ")"), scope, not m_s and not m_a,
         f"{len(names) - len(m_s)}/{len(names)}")
    have = [(k, h) for k, h in names if single_name(k, h) in s]
    badh = [single_name(k, h) for k, h in have if s[single_name(k, h)]["hooks"] != len(sp["window"]) + 1]
    gate("every single head registered window + 1 hooks", scope, not badh, ",".join(badh[:5]))
    worst = max((_maxdiff(s[single_name(k, h)]["ssdc"][:k + 1], s["Cref"]["ssdc"][:k + 1]) for k, h in have), default=0.0)
    gate("single-head masks leave every readout at or before their block bit-identical to Cref", scope, worst <= EXACT_TOL, f"{worst:.1e}")
    inert = [single_name(k, h) for k, h in have if s[single_name(k, h)]["ssdc"][FIN] == s["Cref"]["ssdc"][FIN]]
    gate("every single-head mask changes SSDC at FINAL (no inert mask)", scope, not inert, ",".join(inert[:5]))
    sel = [tuple(x) for x in R.get("axis_heads", [])]
    gate("axis map measured for the heads selected by the fixed rule", scope,
         bool(sel) and sel == select_axis_heads(part, key, R, P, sp) and all(f"{w}_b{k}_h{h}" in s for k, h in sel for w in ("row", "col")),
         f"{sel}")
    session_gate(R, scope)
    return True

## Part e - attention statistics (NpkG Q2; RxJ2 Q3)
W5 part E's recipe, verbatim: first 64 repack images in natural order, batch 32; q and k rebuilt the way the
attention module builds them (rope included; checked against the module's own output); per block and head: mean
attention distance (patches), entropy, mass sent to prefix tokens; plus, new, the distance along rows |dr| and along
columns |dc|. Intact values must equal W5 part E bit for bit. Conditions: the four RoPE models with one axis's planes
(or 16 random planes) off in every block; the hybrid with its rotations off, its embedding zeroed, or both.

In [ ]:
from timm.layers import apply_rot_embed_cat

def attn_qkv(mod, x, rope, npt):                       # W5 part E
    B, N, C = x.shape; nh = mod.num_heads; hd = C // nh
    if getattr(mod, "q_bias", None) is None:
        qkv = mod.qkv(x)
    else:
        bias = torch.cat((mod.q_bias, mod.k_bias, mod.v_bias))
        qkv = (mod.qkv(x) + bias if getattr(mod, "qkv_bias_separate", False) else F.linear(x, mod.qkv.weight, bias))
    q, k, v = qkv.reshape(B, N, 3, nh, hd).permute(2, 0, 3, 1, 4).unbind(0)
    q, k = mod.q_norm(q), mod.k_norm(k)
    if rope is not None:
        half = getattr(mod, "rotate_half", False)
        q = torch.cat([q[:, :, :npt], apply_rot_embed_cat(q[:, :, npt:], rope, half=half)], 2).type_as(v)
        k = torch.cat([k[:, :, :npt], apply_rot_embed_cat(k[:, :, npt:], rope, half=half)], 2).type_as(v)
    return q, k, v

def attn_maps(mod, x, rope, npt):                      # W5 part E
    q, k, v = attn_qkv(mod, x, rope, npt)
    return ((q * mod.scale) @ k.transpose(-2, -1)).softmax(-1)

def attention_stats(model, processor, source, n_prefix, ctx, nh, Hg, Wg):
    blocks = get_vit_blocks(model, source); attns = [get_block_attention(b, source) for b in blocks]
    coords = torch.stack(torch.meshgrid(torch.arange(Hg), torch.arange(Wg), indexing="ij"), -1).reshape(-1, 2).float().to(DEVICE)
    Dm = torch.cdist(coords, coords)
    Dr = (coords[:, None, 0] - coords[None, :, 0]).abs(); Dc = (coords[:, None, 1] - coords[None, :, 1]).abs()
    acc = {i: {k_: torch.zeros(nh, device=DEVICE) for k_ in ("dist", "ent", "pref", "dr", "dc")} | {"n": 0} for i in range(len(blocks))}
    def mk(i):
        def pre(mod, args, kw):
            x = args[0].detach(); rope = kw.get("rope"); npt = int(getattr(mod, "num_prefix_tokens", n_prefix))
            with torch.no_grad():
                att = attn_maps(mod, x, rope, npt)
                ap = att[:, :, npt:, npt:]
                pref = 1.0 - ap.sum(-1)
                ap = ap / ap.sum(-1, keepdim=True).clamp_min(1e-12)
                acc[i]["dist"] += (ap * Dm).sum(-1).mean(-1).sum(0)
                acc[i]["ent"] += (-(ap.clamp_min(1e-12) * ap.clamp_min(1e-12).log()).sum(-1)).mean(-1).sum(0)
                acc[i]["pref"] += pref.mean(-1).sum(0)
                acc[i]["dr"] += (ap * Dr).sum(-1).mean(-1).sum(0)
                acc[i]["dc"] += (ap * Dc).sum(-1).mean(-1).sum(0)
                acc[i]["n"] += x.shape[0]
            return None
        return pre
    hs = []
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            for i, a in enumerate(attns): hs.append(a.register_forward_pre_hook(mk(i), with_kwargs=True))   # after ctx's hooks
            torch.manual_seed(TORCH_SEED)
            dl = prep_data(DS_SSDC, processor, source, corruption_type=None, number_images=E_IMAGES,
                           batch_size=E_BATCH, half=False, num_workers=0)
            with torch.no_grad():
                for im, _ in dl: model(im.to(DEVICE))
    finally:
        for h in hs: h.remove()
    out = {str(i): {"distance": (v["dist"] / max(v["n"], 1)).tolist(), "entropy": (v["ent"] / max(v["n"], 1)).tolist(),
                    "prefix_mass": (v["pref"] / max(v["n"], 1)).tolist(), "dist_row": (v["dr"] / max(v["n"], 1)).tolist(),
                    "dist_col": (v["dc"] / max(v["n"], 1)).tolist()} for i, v in acc.items() if v["n"] > 0}
    return out, hooks, int(acc[0]["n"])

def run_part_e():
    for key in E_MODELS:
        try:
            run_e_model(key)
        except SystemExit:
            raise
        except Exception as ex:
            traceback.print_exc(); gate("part e model ran to the end", f"e:{key}", False, repr(ex)[:300])

def run_e_model(key):
    if True:
        scope = f"e:{key}"
        cfg = {**CFG_BASE, "part": "e", "images": E_IMAGES, "batch": E_BATCH, "rand_planes_seed": RAND_PLANES_SEED}
        R, _ = load_R("e", key, cfg)
        conds = (["intact", "rowoff_all", "coloff_all", "randoff_all"] if key != "rope_ape"
                 else ["intact", "rot_off", "ape_off", "both_off"])
        if R is not None and all(c in R["stats"] for c in conds):
            print(f"\n[{scope}] already complete; re-checking"); check_part_e(key, R); return
        print("\n" + "=" * 78 + f"\n[{scope}] {MODEL_SPECS[key]['name']} (attention statistics)\n" + "=" * 78, flush=True)
        try:
            model, processor, source = load_timm(key, 224); model.eval().float()
        except Exception as ex:
            traceback.print_exc(); gate("model setup completed", scope, False, repr(ex)[:200]); return
        rh, _, nh = attn_attrs(model, source); n_prefix = int(getattr(model, "num_prefix_tokens", 1))
        blocks = get_vit_blocks(model, source); attns = [get_block_attention(b, source) for b in blocks]
        rname, rmod = find_rope(model); emb0 = rmod.get_embed(shape=(14, 14)); n_planes = int(emb0.shape[-1] // 4)
        _, rowp, colp, dec = derive_axis(emb0, n_planes, rh, 14, 14)
        gate("every plane is a row or a column plane, half each", scope, dec and len(rowp) == len(colp) == n_planes // 2)
        rp = sorted(np.random.default_rng(RAND_PLANES_SEED).choice(n_planes, size=n_planes // 2, replace=False).tolist())
        if R is None: R = {"model": key, "part": "e", "config": cfg, "n_heads": int(nh), "row_planes": rowp, "col_planes": colp,
                           "rand_planes": rp, "stats": {}, "hooks": {}, "sessions": []}
        R["sessions"].append(dict(SESSION))
        # W5's exact gate: rebuilt q/k/v through the module's own fused path reproduce its output
        cap = {}; gh = []
        for i, a in enumerate(attns):
            gh.append(a.register_forward_pre_hook(lambda m, args, kw, i=i: cap.__setitem__(("in", i), (args[0].detach(), kw.get("rope"))), with_kwargs=True))
            gh.append(a.register_forward_hook(lambda m, inp, out, i=i: cap.__setitem__(("out", i), out.detach())))
        torch.manual_seed(TORCH_SEED)
        dl = prep_data(DS_SSDC, processor, source, corruption_type=None, number_images=E_BATCH, batch_size=E_BATCH, half=False, num_workers=0)
        with torch.no_grad():
            for im, _ in dl: model(im.to(DEVICE)); break
        for h in gh: h.remove()
        worst = 0.0
        with torch.no_grad():
            for i, a in enumerate(attns):
                x, rope = cap[("in", i)]; out = cap[("out", i)]; B, N, C = x.shape
                npt = int(getattr(a, "num_prefix_tokens", n_prefix))
                q, k, v = attn_qkv(a, x, rope, npt)
                y = a.norm(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))
                g = getattr(a, "gate", None)
                if g is not None: y = y * g(x).sigmoid()
                worst = max(worst, (a.proj(y) - out).abs().max().item())
        cap.clear()
        gate("rebuilt q/k/v reproduce the attention module's output through its own fused path (W5's gate)", scope,
             worst == 0.0 or (TEST_MODE and worst < 1e-5), f"max |delta| {worst:.2e}")
        ctxs = {"intact": NullCtx, "rowoff_all": lambda: RopeGlobal(model, lambda e: mask_planes(e, rowp, n_planes, rh)),
                "coloff_all": lambda: RopeGlobal(model, lambda e: mask_planes(e, colp, n_planes, rh)),
                "randoff_all": lambda: RopeGlobal(model, lambda e: mask_planes(e, rp, n_planes, rh)),
                "rot_off": lambda: RopeGlobal(model, IDEN), "ape_off": lambda: APEZero(model),
                "both_off": lambda: Multi(RopeGlobal(model, IDEN), APEZero(model))}
        for c in conds:
            if c in R["stats"]: continue
            st, hk, n = attention_stats(model, processor, source, n_prefix, ctxs[c](), int(nh), 14, 14)
            R["stats"][c] = st; R["hooks"][c] = {"hooks": hk, "n_images": n}
            print(f"    {c}: blocks 1-4 |dr| {np.mean([np.mean(st[str(b)]['dist_row']) for b in range(1, 5)]):.3f} "
                  f"|dc| {np.mean([np.mean(st[str(b)]['dist_col']) for b in range(1, 5)]):.3f}", flush=True)
        save_R("e", key, R, {}); check_part_e(key, R)
        RESULTS.setdefault("e", {})[key] = R
        del model; gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()

def check_part_e(key, R):
    scope = f"e:{key}"
    exp = {"intact": 0, "rowoff_all": 1, "coloff_all": 1, "randoff_all": 1, "rot_off": 1, "ape_off": 1, "both_off": 2}
    bad = [c for c, v in R["hooks"].items() if v["hooks"] != exp[c] or v["n_images"] != E_IMAGES]
    gate("every condition registered the expected hooks and saw every image", scope, not bad, ",".join(bad))
    if not TEST_MODE:
        w5 = json.loads(FILES[W5_PARTE].read_text())[W5_KEY[key]]
        mine = R["stats"]["intact"]
        same = all(mine[b][f] == w5[b][f] for b in w5 if b != "_meta" for f in ("distance", "entropy", "prefix_mass"))
        gate("intact distance, entropy and prefix mass equal W5 part E bit for bit (every block and head)", scope, same)
    else:
        gate("intact statistics equal W5 part E", scope, True, "test mode: no W5 anchor for tiny models", skip=True)
    session_gate(R, scope)

## Part c - a RoPE encoder trained for a VLM: Qwen2-VL-2B's vision encoder
675M parameters, 32 blocks, 16 heads of 80 dimensions, 14-px patches, axial 2-D RoPE in every block and no
absolute embedding (the authors removed it). At 224×224 it sees a 16×16 grid. Its tokens arrive in 2×2 merge-block
order, so every readout is put back into raster order before SSDC and probes; there is no class token. RPI permutes
the patch-embedding outputs within each image, as the repo's RPI does for ViTs. Its rotary table gives each head 40
planes; the gate checks from the model's own cos/sin that planes 0-19 turn with the row and 20-39 with the column.
No earlier run exists, so the gates are internal. Readings: axis test at block 11 (a third of the depth), rebuild
entering block 31 after rotations are withheld from blocks 0-15, and whether SSDC starts at the floor.

In [ ]:
import transformers.models.qwen2_vl.modeling_qwen2_vl as QM
from transformers import Qwen2VLImageProcessor

def _pnorm(v):
    if hasattr(v, "value"): v = v.value
    if isinstance(v, (list, tuple)): return [round(float(x), 6) for x in v]
    if isinstance(v, float): return round(v, 6)
    return str(v)

def load_qwen():
    info = {"id": QWEN_ID}
    if TEST_MODE:
        from transformers.models.qwen2_vl.configuration_qwen2_vl import Qwen2VLVisionConfig
        torch.manual_seed(301)
        vc = Qwen2VLVisionConfig(depth=32, embed_dim=64, hidden_size=32, num_heads=4, mlp_ratio=2, patch_size=14,
                                 spatial_merge_size=2, temporal_patch_size=2, in_channels=3)
        vis = QM.Qwen2VisionTransformerPretrainedModel._from_config(vc)
        with torch.no_grad():
            for blk in vis.blocks: blk.attn.qkv.weight.mul_(4.0)
        ip = Qwen2VLImageProcessor(min_pixels=C_RES * C_RES, max_pixels=C_RES * C_RES); info["sha"] = "test"
    else:
        from transformers import Qwen2VLForConditionalGeneration
        try:
            from huggingface_hub import HfApi
            info["sha"] = HfApi().model_info(QWEN_ID).sha
        except Exception as ex:
            info["sha"] = f"unavailable ({type(ex).__name__})"
        try:
            full = Qwen2VLForConditionalGeneration.from_pretrained(QWEN_ID, dtype=torch.float32, attn_implementation="sdpa")
        except TypeError:
            full = Qwen2VLForConditionalGeneration.from_pretrained(QWEN_ID, torch_dtype=torch.float32, attn_implementation="sdpa")
        vis = full.model.visual; del full; gc.collect()
        ip_off = Qwen2VLImageProcessor.from_pretrained(QWEN_ID)
        ip = Qwen2VLImageProcessor(min_pixels=C_RES * C_RES, max_pixels=C_RES * C_RES)
        keys = ["image_mean", "image_std", "patch_size", "merge_size", "temporal_patch_size", "resample", "rescale_factor",
                "do_normalize", "do_rescale", "do_convert_rgb"]
        info["processor_diff"] = {k: (_pnorm(getattr(ip, k, None)), _pnorm(getattr(ip_off, k, None))) for k in keys
                                  if _pnorm(getattr(ip, k, None)) != _pnorm(getattr(ip_off, k, None))}
    return vis.to(DEVICE).eval().float(), ip, info

def qwen_cache(dataset, ip, n, bs):
    out, buf, t = [], [], time.time()
    for i, ex in enumerate(dataset):
        if i >= n: break
        buf.append(ex["image"].convert("RGB"))
        if len(buf) == bs:
            f = ip(images=buf, return_tensors="pt"); out.append((f["pixel_values"].clone(), f["image_grid_thw"].clone())); buf = []
    if buf:
        f = ip(images=buf, return_tensors="pt"); out.append((f["pixel_values"].clone(), f["image_grid_thw"].clone()))
    print(f"    cached {sum(int(g.shape[0]) for _, g in out)} images in {time.time() - t:.0f}s", flush=True)
    return out

class QwenRot:
    """Rotation planes off in chosen blocks: cos -> 1, sin -> 0 on the plane's two columns (rotate-half layout)."""
    def __init__(s, vis, blocks, planes, half):
        s.vis, s.blocks, s.half = vis, list(blocks), half
        s.cols = torch.tensor(sorted([p for p in planes] + [p + half for p in planes]), dtype=torch.long)
        s.handles = []
    def __enter__(s):
        cols = s.cols
        def pre(mod, args, kw):
            pe = kw.get("position_embeddings")
            if pe is None: raise RuntimeError("Qwen attention called without position_embeddings")
            cos, sin = pe; c = cols.to(cos.device)
            cos = cos.clone(); sin = sin.clone(); cos[..., c] = 1.0; sin[..., c] = 0.0
            kw = dict(kw); kw["position_embeddings"] = (cos, sin); return args, kw
        for i in s.blocks: s.handles.append(s.vis.blocks[i].attn.register_forward_pre_hook(pre, with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

def measure_qwen(M, ctx, capture=(), cache=None, rpi=True):
    vis, T, order, NB = M["model"], M["T"], M["order"], M["n_blocks"]; FIN = NB
    acc = SSDCAccumulator(n_prefix=0, spatial_metric=METRIC); cap = {r: [] for r in capture}; st = {"n": 0, "perm": None}
    def rec(i, x):
        t = x.detach().view(-1, T, x.shape[-1])[:, order]
        acc.add(i, t)
        if i in cap: cap[i].append(t.float().cpu())
    hs = [blk.attn.register_forward_hook(lambda m, inp, out, i=i: rec(i, inp[0])) for i, blk in enumerate(vis.blocks)]
    hs.append(vis.merger.ln_q.register_forward_hook(lambda m, inp, out: rec(FIN, out)))
    if rpi:
        def rpi_hook(m, inp, out):
            B = out.shape[0] // T; perm = torch.randperm(T); st["perm"] = perm
            return out.view(B, T, -1)[:, perm].reshape(B * T, -1)
        hs.append(vis.patch_embed.register_forward_hook(rpi_hook))
    t0 = time.time()
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            with torch.inference_mode():
                for pv, g in (cache if cache is not None else M["cache"]):
                    vis(hidden_states=pv.to(DEVICE), grid_thw=g.to(DEVICE)); st["n"] += int(g.shape[0])
    finally:
        for h in hs: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(NB)] + [acc.ssdc(FIN, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": st["n"], "runtime_s": round(time.time() - t0, 1)}
    if capture: res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

def setup_qwen():
    scope = "c:qwen2vl"
    print("\n" + "=" * 78 + f"\n[{scope}] {QWEN_ID} vision encoder at {C_RES}x{C_RES}\n" + "=" * 78, flush=True)
    vis, ip, info = load_qwen(); cfg_v = vis.config
    head_dim = cfg_v.embed_dim // cfg_v.num_heads; half = head_dim // 2; n_planes = half
    M = {"model": vis, "processor": ip, "info": info, "n_blocks": len(vis.blocks), "n_heads": cfg_v.num_heads,
         "head_dim": head_dim, "half": half, "n_planes": n_planes, "scope": scope}
    if not TEST_MODE:
        gate("vision config is Qwen2-VL's 675M encoder (32 blocks, width 1280, 16 heads, patch 14, merge 2)", scope,
             (len(vis.blocks), cfg_v.embed_dim, cfg_v.num_heads, cfg_v.patch_size, cfg_v.spatial_merge_size) == (32, 1280, 16, 14, 2),
             f"{len(vis.blocks)} blocks, width {cfg_v.embed_dim}, {cfg_v.num_heads} heads, patch {cfg_v.patch_size}")
        gate("parameter count of the vision encoder is about 675M", scope,
             600e6 < sum(p.numel() for p in vis.parameters()) < 700e6, f"{sum(p.numel() for p in vis.parameters()) / 1e6:.1f}M; revision {info['sha']}")
    pos_named = [n for n, _ in vis.named_parameters() if "pos_embed" in n or "position_embedding" in n]
    gate("no learned absolute position embedding (RoPE only)", scope, not pos_named, ",".join(pos_named))
    M["cache"] = qwen_cache(DS_SSDC, ip, N_SSDC, C_BATCH)
    grids = torch.cat([g for _, g in M["cache"]], 0)
    G = C_RES // cfg_v.patch_size
    gate(f"every image becomes a {G}x{G} grid (no dynamic resizing)", scope,
         bool((grids == torch.tensor([1, G, G])).all()), f"{grids[0].tolist()} x {grids.shape[0]}")
    if not TEST_MODE:
        gate("our fixed-size processor equals the checkpoint's own processor in every setting except the image size", scope,
             not info.get("processor_diff"), str(info.get("processor_diff")))
    gate("image normalisation is the checkpoint's (CLIP mean/std)", scope,
         [round(v, 4) for v in ip.image_mean] == [0.4815, 0.4578, 0.4082], f"{ip.image_mean}")
    pos = QM.get_vision_position_ids(torch.tensor([[1, G, G]]), cfg_v.spatial_merge_size)
    ridx = pos[:, 0] * G + pos[:, 1]
    gate("position ids cover the grid once (merge-block order is a permutation of raster order)", scope,
         sorted(ridx.tolist()) == list(range(G * G)) and tuple(pos.shape) == (G * G, 2))
    M["order"] = torch.argsort(ridx).to(DEVICE); M["T"] = G * G; M["G"] = G
    with torch.no_grad():
        cos, sin = vis.rotary_pos_emb(torch.zeros(1, device=DEVICE), pos.to(DEVICE))
    gate("rotary table: one cos/sin column per head dimension, shared by all heads", scope,
         tuple(cos.shape) == (G * G, head_dim), f"{tuple(cos.shape)}")
    cr = cos[M["order"]].float(); sr = sin[M["order"]].float(); rows = []; dec = True
    for p in range(n_planes):
        z = torch.complex(cr[:, p], sr[:, p]).reshape(G, G)
        ra = (z - z.mean(dim=1, keepdim=True)).abs().mean().item(); rb = (z - z.mean(dim=0, keepdim=True)).abs().mean().item()
        ax = "row" if ra < rb else "col"; lo, hi = sorted([ra, rb]); r = lo / hi if hi > 1e-12 else 1.0
        dec &= r <= 0.10; rows.append(ax)
        same = torch.equal(cos[:, p], cos[:, p + half]) and torch.equal(sin[:, p], sin[:, p + half])
        dec &= same
    rowp = [p for p in range(n_planes) if rows[p] == "row"]; colp = [p for p in range(n_planes) if rows[p] == "col"]
    gate("every plane turns with the row only or the column only; planes 0-19 row, 20-39 column (from the model's own cos/sin)", scope,
         dec and rowp == list(range(n_planes // 2)) and colp == list(range(n_planes // 2, n_planes)), f"{len(rowp)} row / {len(colp)} col")
    M["row_planes"], M["col_planes"] = rowp, colp
    M["rand_planes"] = sorted(np.random.default_rng(RAND_PLANES_SEED).choice(n_planes, size=n_planes // 2, replace=False).tolist())
    # algebra: an empty mask is a no-op; masking every plane removes the rotation from q and k
    q = torch.randn(5, cfg_v.num_heads, head_dim, device=DEVICE)
    qa, _ = QM.apply_rotary_pos_emb_vision(q, q, cos[:5], sin[:5])
    c1, s1 = cos[:5].clone(), sin[:5].clone(); c1[:] = 1.0; s1[:] = 0.0
    qi, _ = QM.apply_rotary_pos_emb_vision(q, q, c1, s1)
    gate("with every plane masked the rotation is the identity", scope, torch.equal(qi, q) and not torch.equal(qa, q))
    pv, g = M["cache"][0]
    with torch.no_grad():
        o0 = vis(hidden_states=pv[: 2 * G * G].to(DEVICE), grid_thw=g[:2].to(DEVICE)).last_hidden_state
        with QwenRot(vis, range(len(vis.blocks)), [], half): o1 = vis(hidden_states=pv[: 2 * G * G].to(DEVICE), grid_thw=g[:2].to(DEVICE)).last_hidden_state
        with QwenRot(vis, range(len(vis.blocks)), list(range(n_planes)), half): o2 = vis(hidden_states=pv[: 2 * G * G].to(DEVICE), grid_thw=g[:2].to(DEVICE)).last_hidden_state
    gate("an empty plane mask leaves the encoder's output bit-identical; masking all planes changes it", scope,
         torch.equal(o0, o1) and not torch.equal(o0, o2))
    conv = vis.patch_embed.proj; grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    with torch.no_grad(): vis(hidden_states=pv.to(DEVICE), grid_thw=g.to(DEVICE))
    h.remove()
    with torch.no_grad():
        y64 = F.conv3d(grab["x"].double(), conv.weight.double(), None, conv.stride)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    M["first_layer_rel_diff"] = v; del grab, y64
    gate("patch embedding (Conv3d) agrees with a float64 computation to TF32 level or better", scope, v <= FIRST_LAYER_RANGE[1], f"{v:.3e}")
    a = measure_qwen(M, NullCtx())["ssdc"]
    b = measure_qwen(M, NullCtx(), cache=qwen_cache(DS_SSDC, ip, N_SSDC, C_BATCH))["ssdc"]
    gate("intact SSDC from the cache equals a fresh decode of the same images", scope, a == b, f"max |delta| {_maxdiff(a, b):.1e}")
    nr = measure_qwen(M, NullCtx(), rpi=False)["ssdc"]
    gate("RPI changes the measurement (the shuffle is applied)", scope, nr != a)
    return M

def run_part_c():
    scope = "c:qwen2vl"
    cfg = {**CFG_BASE, "part": "c", "model": QWEN_ID, "res": C_RES, "batch": C_BATCH, "windows": C_WINDOWS,
           "axis_readout": C_AXIS_READOUT, "probes_base": C_PROBES_BASE, "rand_planes_seed": RAND_PLANES_SEED}
    R, _ = load_R("c", "qwen2vl", cfg)
    names = ["baseline", "floor", "rowoff_all", "coloff_all", "randoff_all"] + list(C_WINDOWS) + ["Dref_row", "Dref_col"]
    if R is not None and all(n in R["ssdc"] for n in names):
        print(f"\n[{scope}] already complete; re-checking"); check_part_c(R); RESULTS["c"] = R; return
    try:
        M = setup_qwen()
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", scope, False, repr(ex)[:300]); return
    if scope_failed(scope) and STRICT and not TEST_MODE:
        gate("part c measured", scope, False, "STOPPED: a setup gate failed (RB7_STRICT=1)"); release(M); return
    vis, NB, half, G = M["model"], M["n_blocks"], M["half"], M["G"]; FIN, last = NB, NB - 1
    if R is None:
        R = {"model": "qwen2vl", "name": QWEN_ID, "revision": M["info"].get("sha"), "part": "c", "config": cfg, "n_heads": M["n_heads"],
             "n_blocks": NB, "grid": [G, G], "row_planes": M["row_planes"], "col_planes": M["col_planes"], "rand_planes": M["rand_planes"],
             "readouts": f"0-{last} = input of block b after norm1 (raster order); {FIN} = merger.ln_q output (before the 2x2 merge)",
             "ssdc": {}, "sessions": []}
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M["first_layer_rel_diff"], "revision": M["info"].get("sha")})
    allb = list(range(NB)); P_ = lambda pl: list(pl)
    arms = {"baseline": (NullCtx, C_PROBES_BASE + [FIN], ("exact", "row", "col")),
            "floor": (lambda: QwenRot(vis, allb, list(range(M["n_planes"])), half), C_PROBES_BASE + [FIN], ("exact", "row", "col")),
            "rowoff_all": (lambda: QwenRot(vis, allb, M["row_planes"], half), [4, C_AXIS_READOUT, 16], ("exact", "row", "col")),
            "coloff_all": (lambda: QwenRot(vis, allb, M["col_planes"], half), [4, C_AXIS_READOUT, 16], ("exact", "row", "col")),
            "randoff_all": (lambda: QwenRot(vis, allb, M["rand_planes"], half), [4, C_AXIS_READOUT, 16], ("exact", "row", "col"))}
    for wn, wb in C_WINDOWS.items():
        arms[wn] = ((lambda wb=wb: QwenRot(vis, wb, list(range(M["n_planes"])), half)),
                    ([last, FIN] if wn == C_MAIN else []), ("exact", "row", "col"))
    arms["Dref_row"] = (lambda: QwenRot(vis, C_WINDOWS[C_MAIN], M["row_planes"], half), [last, FIN], ("row", "col"))
    arms["Dref_col"] = (lambda: QwenRot(vis, C_WINDOWS[C_MAIN], M["col_planes"], half), [last, FIN], ("row", "col"))
    t0 = time.time()
    for name, (mk, ro, heads) in arms.items():
        if name in R["ssdc"]: continue
        r = measure_qwen(M, mk(), capture=tuple(ro))
        pr = {str(b): probe_block(r["acts"][b], G, G, b, heads=heads) for b in ro}
        R["ssdc"][name] = {"ssdc": r["ssdc"], "hooks": r["hooks"], "n_images": r["n_images"], "probes": pr}
        del r; save_R("c", "qwen2vl", R, {})
        print(f"    {name}: SSDC entering block {last} {R['ssdc'][name]['ssdc'][last]:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
    R["complete"] = all(n in R["ssdc"] for n in names); save_R("c", "qwen2vl", R, {})
    check_part_c(R); RESULTS["c"] = R
    gate("part c finished", scope, True, f"{(time.time() - t0) / 60:.1f} min; complete {R['complete']}")
    release(M)

def check_part_c(R):
    scope = "c:qwen2vl"; s = R["ssdc"]; NB = R["n_blocks"]; last = NB - 1
    exp = {"baseline": 0, "floor": NB, "rowoff_all": NB, "coloff_all": NB, "randoff_all": NB, "Dref_row": len(C_WINDOWS[C_MAIN]),
           "Dref_col": len(C_WINDOWS[C_MAIN])} | {w: len(b) for w, b in C_WINDOWS.items()}
    bad = [n for n in s if s[n]["hooks"] != exp[n] or s[n]["n_images"] != N_SSDC]
    gate("every arm registered the expected hooks and saw every image", scope, not bad, ",".join(bad))
    _d = f"{s['baseline']['ssdc'][last]:.4f} -> {s['floor']['ssdc'][last]:.4f}"
    if TEST_MODE: gate("floor lowers SSDC entering the last block", scope, True, "test mode: " + _d, skip=True)
    else: gate("floor lowers SSDC entering the last block", scope, s["floor"]["ssdc"][last] < s["baseline"]["ssdc"][last], _d)
    pre = max(_maxdiff(s[n]["ssdc"][:1], s["baseline"]["ssdc"][:1]) for n in s)
    gate("every arm leaves readout 0 bit-identical to intact (masks act inside attention)", scope, pre <= EXACT_TOL, f"{pre:.1e}")
    session_gate(R, scope)

## Measurement
Parts run in the order d, a, b, e, c. Each (part, model) saves after every few arms; Run All again after a stop and
it continues where it stopped. A reference-gate failure stops that model (`RB7_STRICT=1`); the other models and
parts still run.

In [ ]:
RESULTS = {}
T_RUN = time.time()
for part in PARTS:
    t_p = time.time()
    print("\n" + "#" * 78 + f"\n# PART {part}\n" + "#" * 78, flush=True)
    try:
        if part == "d": run_part_d()
        elif part == "a":
            for key in A_MODELS:
                try:
                    RESULTS.setdefault("a", {})[key] = run_scale("a", key)
                except SystemExit:
                    raise
                except Exception as ex:
                    traceback.print_exc(); gate("part a model ran to the end", f"a:{key}", False, repr(ex)[:300])
        elif part == "b": RESULTS.setdefault("b", {})["large"] = run_scale("b", "large")
        elif part == "e": run_part_e()
        elif part == "c": run_part_c()
    except SystemExit:
        raise
    except Exception as ex:
        traceback.print_exc(); gate(f"part {part} ran to the end", part, False, repr(ex)[:300])
    print(f"\npart {part}: {(time.time() - t_p) / 60:.1f} min", flush=True)
print(f"\nmeasurement finished in {(time.time() - T_RUN) / 60:.1f} min")

## Analysis
Reads only the saved files, so it can be re-run on its own. Every verdict comes from the rules in the CONFIG cell.

In [ ]:
from scipy.stats import rankdata
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
def show(title, obj):
    print("\n" + title); print("-" * len(title))
    if isinstance(obj, pd.DataFrame): print(obj.to_string(index=False) if len(obj) else "(empty)")
    else: print(json.dumps(obj, indent=1, default=float))
def spearman(x, y):
    rx, ry = rankdata(x), rankdata(y)
    return float("nan") if (np.std(rx) == 0 or np.std(ry) == 0) else float(np.corrcoef(rx, ry)[0, 1])
def within_block(blocks, x, y, n_perm, seed=0):
    blocks, x, y = np.asarray(blocks), np.asarray(x, float), np.asarray(y, float); ub = sorted(set(blocks.tolist()))
    per = {int(b): spearman(x[blocks == b], y[blocks == b]) for b in ub}
    stat = float(np.nanmean(list(per.values()))); g = np.random.default_rng(seed); null = []
    for _ in range(n_perm):
        yy = y.copy()
        for b in ub:
            idx = np.where(blocks == b)[0]; yy[idx] = y[g.permutation(idx)]
        null.append(np.nanmean([spearman(x[blocks == b], yy[blocks == b]) for b in ub]))
    p = float((np.sum(np.abs(null) >= abs(stat)) + 1) / (n_perm + 1))
    return {"per_block": per, "mean": stat, "p": p}
def paired_pts(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)
def Nb(v, base, flo, b):
    d = base[b] - flo[b]; return (v[b] - flo[b]) / d if abs(d) > 1e-12 else float("nan")
def retn(arm, base, flo):
    d = base - flo; return (arm - flo) / d if abs(d) > 1e-9 else float("nan")
def axis_verdict(rr_row, rr_col, cr_col, cr_row):
    if rr_row <= AXIS_GONE and rr_col >= AXIS_KEPT and cr_col <= AXIS_GONE and cr_row >= AXIS_KEPT: return "AXES SEPARATE"
    if rr_row < rr_col and cr_col < cr_row: return "PARTLY SEPARATE"
    return "NOT SEPARATE"
def rebuild_verdict(n):
    return "REBUILDS" if n >= REBUILD_HI else ("REBUILDS LESS" if n >= REBUILD_LO else "DOES NOT REBUILD")
def label(rc, cc, bar):        # RB-2
    if rc > bar and rc - cc > bar: return "row"
    if cc > bar and cc - rc > bar: return "col"
    if rc > bar or cc > bar: return "mixed"
    return "none"
def f2(x, n=2): return "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{n}f}"
def load_part(part, key):
    f = rfile(part, key)
    if not f.exists(): return None, {}
    R = json.loads(f.read_text())
    P = {n: z_[n] for z_ in [np.load(pfile(part, key))] for n in z_.files} if pfile(part, key).exists() else {}
    return R, P
ANALYSIS, TABLES, REPORT = {}, {}, []
def add_table(name, rows): TABLES.setdefault(name, []).extend(rows)

def axis_test(s, r, arms=("rowoff_all", "coloff_all", "randoff_all")):
    b, f = s["baseline"]["probes"][str(r)], s["floor"]["probes"][str(r)]
    out = {}
    for a in arms:
        p = s[a]["probes"][str(r)]
        out[a] = {"row": p["row"], "col": p["col"], "row_ret": retn(p["row"], b["row"], f["row"]), "col_ret": retn(p["col"], b["col"], f["col"])}
    out["intact"] = {"row": b["row"], "col": b["col"]}; out["floor"] = {"row": f["row"], "col": f["col"]}
    out["verdict"] = axis_verdict(out["rowoff_all"]["row_ret"], out["rowoff_all"]["col_ret"], out["coloff_all"]["col_ret"], out["coloff_all"]["row_ret"])
    return out

# part d
def analyse_d():
    out = {}
    for key in D_MODELS:
        R, P = load_part("d", key)
        if not R: continue
        cls = MODEL_SPECS[key]["classifier"]; s = R["ssdc"]; cref = s["Cref"]["ssdc"]
        nulls = [n for n in s if n.startswith("null")]
        o = {"top6": R["config"]["top6"]}
        sc = {n: cref[11] - s[n]["ssdc"][11] for n in nulls + ["top6"]}
        ns = np.array([sc[n] for n in nulls]); below = int((ns < sc["top6"]).sum())
        o["ssdc"] = {"top6_cost": sc["top6"], "null_max": float(ns.max()), "null_mean": float(ns.mean()), "n": len(ns), "n_below": below,
                     "verdict": "HOLDS" if below == len(ns) else ("HOLDS AT 95%" if below >= int(np.ceil(0.95 * len(ns))) else "DOES NOT HOLD")}
        D0 = RB0[key]["D"]; s1 = np.array([cref[11] - D0["ssdc"][f"rand{j:02d}"][11] for j in range(len(D0["random_sets"]))])
        o["ssdc"]["one_per_block_n_below"] = [int((s1 < sc["top6"]).sum()), len(s1)]
        if cls and "window" in R["acc"]:
            w = R["acc"]["window"]["acc"]; tc = {n: (w - R["acc"][n]["acc"]) * 100 for n in nulls + ["top6"] if n in R["acc"]}
            na = np.array([tc[n] for n in nulls if n in tc]); below = int((na < tc["top6"]).sum())
            m_, se_ = paired_pts(P["window"], P["top6"])
            o["top1"] = {"top6_cost": tc["top6"], "top6_se": se_, "null_max": float(na.max()), "null_mean": float(na.mean()),
                         "null_p95": float(np.percentile(na, 95)), "n": len(na), "n_below": below,
                         "verdict": "HOLDS" if below == len(na) else ("HOLDS AT 95%" if below >= int(np.ceil(0.95 * len(na))) else "DOES NOT HOLD")}
            a1 = np.array([(w - D0["acc"][f"rand{j:02d}"]) * 100 for j in range(24) if f"rand{j:02d}" in D0["acc"]])
            o["top1"]["one_per_block_n_below"] = [int((a1 < tc["top6"]).sum()), len(a1)]
            A = RB0[key]["A"]["conditions"]; C = RB0[key]["C"]["conditions"]; nh = R["n_heads"]; win0 = C["window"]["acc"]
            rows = [(k, A["Cref"]["ssdc"][11] - A[single_name(k, h)]["ssdc"][11], (win0 - C[single_name(k, h)]["acc"]) * 100)
                    for k in range(6, 11) for h in range(nh)]
            bl, x, y = zip(*rows)
            wb = within_block(bl, x, y, D_PERM_N)
            o["ssdc_vs_top1"] = {"pooled_rho": spearman(x, y), "within_block_mean_rho": wb["mean"], "within_block_p": wb["p"],
                                 "per_block": wb["per_block"]}
            add_table("d_null", [{"model": key, "metric": "top1_pts", "set": n, "cost": tc[n]} for n in tc])
        add_table("d_null", [{"model": key, "metric": "ssdc_b11", "set": n, "cost": sc[n]} for n in sc])
        out[key] = o
    return out

# parts a, b
def analyse_scale(part, key):
    R, P = load_part(part, key)
    if not R or "Cref" not in R["ssdc"]: return None
    sp = scale_spec(part, key); s = R["ssdc"]; NB = R["n_blocks"]; FIN, last = NB, NB - 1; nh = R["n_heads"]
    cls = MODEL_SPECS[key]["classifier"]; base, flo = s["baseline"]["ssdc"], s["floor"]["ssdc"]
    o = {"axis_test": axis_test(s, sp["axis_r"]), "axis_readout": sp["axis_r"]}
    o["rebuild"] = {"Cref": Nb(s["Cref"]["ssdc"], base, flo, last)}
    for wn in sp["extra"]: o["rebuild"][wn] = Nb(s[wn]["ssdc"], base, flo, last)
    o["rebuild"]["verdict"] = rebuild_verdict(o["rebuild"]["Cref"])
    o["ssdc_intact_last"], o["ssdc_floor_last"] = base[last], flo[last]
    if cls and "window" in R["acc"]:
        o["top1_refs"] = {n: R["acc"][n]["acc"] * 100 for n in ("baseline", "floor", "window")}
    have = [(k, h) for k in sp["post"] for h in range(nh) if single_name(k, h) in s]
    if not have: return o
    sr = ssdc_rank(R, sp); o["top_ssdc_head"] = list(sr[0][:2]) if sr else None
    o["top_ssdc_head_in_first_two_blocks"] = bool(sr and sr[0][0] in sp["post"][:2])
    r11 = sorted([(s["Cref"]["ssdc"][FIN] - s[single_name(last, h)]["ssdc"][FIN], h) for h in range(nh) if single_name(last, h) in s], reverse=True)
    o["top_ssdc_head_last_block"] = [last, r11[0][1]] if r11 else None
    rows = []
    t1 = {}
    if cls and "window" in R["acc"]:
        for k, h in have:
            if single_name(k, h) in P:
                m_, se_ = paired_pts(P["window"], P[single_name(k, h)]); t1[(k, h)] = (m_, se_)
    for k, h in have:
        rows.append({"model": key, "part": part, "block": k, "head": h,
                     "ssdc_cost_last": s["Cref"]["ssdc"][last] - s[single_name(k, h)]["ssdc"][last] if k < last else None,
                     "ssdc_cost_final": s["Cref"]["ssdc"][FIN] - s[single_name(k, h)]["ssdc"][FIN],
                     "top1_cost": t1.get((k, h), (None, None))[0], "top1_se": t1.get((k, h), (None, None))[1]})
    if t1:
        tr = sorted(t1.items(), key=lambda kv: (-kv[1][0], kv[0])); rank = {kh: i + 1 for i, (kh, _) in enumerate(tr)}
        o["costliest_top1_head"] = [*tr[0][0], tr[0][1][0], tr[0][1][1]]
        o["costliest_top1_head_in_first_two_blocks"] = tr[0][0][0] in sp["post"][:2]
        pos = {kh: v[0] for kh, v in t1.items() if v[0] > 0}; tot = sum(pos.values())
        o["share_top1_first_two_blocks"] = sum(v for kh, v in pos.items() if kh[0] in sp["post"][:2]) / tot if tot > 0 else float("nan")
        xs = [(k, s["Cref"]["ssdc"][last] - s[single_name(k, h)]["ssdc"][last], t1[(k, h)][0]) for k, h in have if k < last and (k, h) in t1]
        bl, x, y = zip(*xs)
        wb = within_block(bl, x, y, D_PERM_N)
        o["ssdc_vs_top1"] = {"pooled_rho": spearman(x, y), "within_block_mean_rho": wb["mean"], "within_block_p": wb["p"]}
        for r in rows: r["top1_rank"] = rank.get((r["block"], r["head"]))
    add_table(f"{part}_singles", rows)
    # axis map with RB-2's noise bar
    sel = [tuple(x) for x in R.get("axis_heads", [])]; amap = []
    def sd(n, r, ax): return s[n]["probes"][str(r)].get("noise", {}).get(ax, {}).get("combined_sd")
    bars = {r: 300 * max(sd("Dref_row", r, "row"), sd("Dref_col", r, "col")) for r in (last, FIN)}
    for k, h in sel:
        if f"row_b{k}_h{h}" not in s or f"col_b{k}_h{h}" not in s: continue
        r = last if k < last else FIN
        rc = (s["Dref_row"]["probes"][str(r)]["row"] - s[f"row_b{k}_h{h}"]["probes"][str(r)]["row"]) * 100
        cc = (s["Dref_col"]["probes"][str(r)]["col"] - s[f"col_b{k}_h{h}"]["probes"][str(r)]["col"]) * 100
        amap.append({"model": key, "part": part, "block": k, "head": h, "readout": "last" if r == last else "final",
                     "row_cost_pts": rc, "col_cost_pts": cc, "bar_pts": bars[r], "favours": label(rc, cc, bars[r]),
                     "top1_cost": t1.get((k, h), (None,))[0]})
    add_table(f"{part}_axis", amap)
    above = [a for a in amap if a["favours"] != "none"]
    o["axis_map"] = {"n_mapped": len(amap), "n_above_noise": len(above), "n_one_axis": sum(a["favours"] in ("row", "col") for a in above),
                     "bars_pts": {"last": bars[last], "final": bars[FIN]}, "heads": amap}
    if part == "a":
        rh = ref_head_of(key, nh); c224 = top1_224(key)
        r224 = sorted(c224.items(), key=lambda kv: (-kv[1], kv[0])); rk224 = {kh: i + 1 for i, (kh, _) in enumerate(r224)}
        lab224 = None
        if RB2MAP is not None:
            m_ = RB2MAP[(RB2MAP.model == key) & (RB2MAP.block == rh[0]) & (RB2MAP["head"] == rh[1])]
            lab224 = str(m_.iloc[0]["favours"]) if len(m_) else None
        lab320 = next((a["favours"] for a in amap if (a["block"], a["head"]) == rh), None)
        rk320 = {kh: i + 1 for i, (kh, _) in enumerate(sorted(t1.items(), key=lambda kv: (-kv[1][0], kv[0])))} if t1 else {}
        common = [kh for kh in c224 if kh in t1]
        A0 = RB0[key]["A"]["conditions"]
        pos224 = {kh: v for kh, v in c224.items() if v > 0}; tot224 = sum(pos224.values())
        o["ref_head"] = {"head": list(rh), "rank_320": rk320.get(rh), "of": len(t1), "cost_320": t1.get(rh, (None, None))[0],
                         "se_320": t1.get(rh, (None, None))[1], "rank_224": rk224.get(rh), "cost_224": c224.get(rh),
                         "label_224": lab224, "label_320": lab320,
                         "keeps_rank": (rk320.get(rh) or 999) <= 3, "keeps_axis": (lab224 in ("row", "col") and lab320 == lab224)}
        o["ref_head"]["verdict"] = ("KEEPS ITS RANK" if o["ref_head"]["keeps_rank"] else "MOVES") + " / " + \
                                   ("KEEPS ITS AXIS" if o["ref_head"]["keeps_axis"] else "AXIS CHANGES OR NOT MAPPED")
        o["top1_224_vs_320_rho"] = spearman([c224[kh] for kh in common], [t1[kh][0] for kh in common]) if common else float("nan")
        o["share_top1_first_two_blocks_224"] = sum(v for kh, v in pos224.items() if kh[0] in (6, 7)) / tot224 if tot224 > 0 else float("nan")
        o["rebuild_224"] = Nb(A0["Cref"]["ssdc"], A0["baseline"]["ssdc"], A0["floor"]["ssdc"], 11)
    return o

# part e
def analyse_e():
    out = {"rope": {}, "hybrid": None}; cases = []
    for key in E_MODELS:
        R, _ = load_part("e", key)
        if not R: continue
        st = R["stats"]; avg = lambda c, f, bl: float(np.mean([np.mean(st[c][str(b)][f]) for b in bl]))
        if key != "rope_ape":
            o = {c: {"dr": avg(c, "dist_row", range(1, 5)), "dc": avg(c, "dist_col", range(1, 5)), "dist": avg(c, "distance", range(1, 5)),
                     "entropy": avg(c, "entropy", range(1, 5))} for c in st}
            o["widen_row_when_row_off"] = o["rowoff_all"]["dr"] - o["intact"]["dr"]
            o["widen_col_when_col_off"] = o["coloff_all"]["dc"] - o["intact"]["dc"]
            o["widen_random_dr_dc"] = [o["randoff_all"]["dr"] - o["intact"]["dr"], o["randoff_all"]["dc"] - o["intact"]["dc"]]
            rise_r = rise_c = None
            if not TEST_MODE and key in W5_PARTA and W5_PARTA[key] in FILES:
                w5 = json.loads(FILES[W5_PARTA[key]].read_text())["conditions"]
                b4 = lambda n: w5[n]["probes"]["4"]
                rise_c = (b4("axis_row")["col"] - b4("baseline")["col"]) * 100     # row planes off: column probe vs intact
                rise_r = (b4("axis_col")["row"] - b4("baseline")["row"]) * 100     # column planes off: row probe vs intact
            o["rise_col_probe_when_row_off_pts"], o["rise_row_probe_when_col_off_pts"] = rise_c, rise_r
            cases += [(key, "row", o["widen_row_when_row_off"], rise_c), (key, "col", o["widen_col_when_col_off"], rise_r)]
            out["rope"][key] = o
        else:
            o = {c: {"dist": [float(np.mean(st[c][str(b)]["distance"])) for b in range(4, 12)],
                     "entropy": [float(np.mean(st[c][str(b)]["entropy"])) for b in range(4, 12)]} for c in st}
            d_rot = np.array(o["rot_off"]["dist"]) - np.array(o["intact"]["dist"]); d_ape = np.array(o["ape_off"]["dist"]) - np.array(o["intact"]["dist"])
            o["rot_off_dist_increase_mean"] = float(d_rot.mean()); o["ape_off_dist_change_mean"] = float(d_ape.mean())
            o["rot_off_blocks_wider"] = int((d_rot > 0).sum())
            o["rot_off_entropy_change_mean"] = float((np.array(o["rot_off"]["entropy"]) - np.array(o["intact"]["entropy"])).mean())
            o["ape_off_entropy_change_mean"] = float((np.array(o["ape_off"]["entropy"]) - np.array(o["intact"]["entropy"])).mean())
            o["verdict"] = ("ROTATIONS KEEP LATE ATTENTION LOCAL" if (o["rot_off_blocks_wider"] >= 6 and o["rot_off_dist_increase_mean"] >= 0.25
                            and abs(o["ape_off_dist_change_mean"]) < 0.5 * o["rot_off_dist_increase_mean"]) else "NOT SHOWN")
            out["hybrid"] = o
        for c in st:
            for b in st[c]:
                add_table("e_attention", [{"model": key, "condition": c, "block": int(b), "head": h, "distance": st[c][b]["distance"][h],
                                           "dist_row": st[c][b]["dist_row"][h], "dist_col": st[c][b]["dist_col"][h],
                                           "entropy": st[c][b]["entropy"][h], "prefix_mass": st[c][b]["prefix_mass"][h]}
                                          for h in range(len(st[c][b]["distance"]))])
    if cases:
        n_w = sum(c[2] > 0 for c in cases); rises = [c for c in cases if c[3] is not None]
        rho = spearman([c[2] for c in rises], [c[3] for c in rises]) if len(rises) >= 3 else float("nan")
        verdict = ("WIDER POOLING SUPPORTED" if (n_w >= 7 and np.isfinite(rho) and rho > 0.5) else
                   ("NOT SUPPORTED" if n_w <= 4 else "MIXED"))
        out["E1"] = {"cases": [{"model": c[0], "removed_axis": c[1], "widening_pts": c[2], "other_probe_rise_pts": c[3]} for c in cases],
                     "n_widening": n_w, "n_cases": len(cases), "spearman_widening_vs_rise": rho, "verdict": verdict}
    return out

# part c
def analyse_c():
    R, _ = load_part("c", "qwen2vl")
    if not R or "baseline" not in R["ssdc"]: return None
    s = R["ssdc"]; NB = R["n_blocks"]; FIN, last = NB, NB - 1; base, flo = s["baseline"]["ssdc"], s["floor"]["ssdc"]
    o = {"revision": R.get("revision"), "ssdc_intact_0": base[0], "ssdc_floor_0": flo[0], "ssdc_intact_last": base[last],
         "ssdc_floor_last": flo[last]}
    o["start_verdict"] = "STARTS NEAR ZERO" if base[0] <= 0.05 else "DOES NOT START NEAR ZERO"
    pk = int(np.argmax(base[:NB])); o["peak_ssdc"] = {"value": base[pk], "block": pk}
    o["first_block_half_peak"] = next((b for b in range(NB) if base[b] >= 0.5 * base[pk]), None)
    if all(n in s for n in ("rowoff_all", "coloff_all", "randoff_all")):
        o["axis_test"] = {str(r): axis_test(s, r) for r in [4, C_AXIS_READOUT, 16]}
        o["axis_verdict"] = o["axis_test"][str(C_AXIS_READOUT)]["verdict"]
    o["rebuild"] = {w: Nb(s[w]["ssdc"], base, flo, last) for w in C_WINDOWS if w in s}
    if C_MAIN in o["rebuild"]: o["rebuild_verdict"] = rebuild_verdict(o["rebuild"][C_MAIN])
    if all(n in s for n in ("Dref_row", "Dref_col")):
        b31, f31 = s["baseline"]["probes"][str(last)], s["floor"]["probes"][str(last)]
        o["one_axis_window"] = {"row_off_0_15": {"row_ret": retn(s["Dref_row"]["probes"][str(last)]["row"], b31["row"], f31["row"]),
                                                 "col_ret": retn(s["Dref_row"]["probes"][str(last)]["col"], b31["col"], f31["col"])},
                                "col_off_0_15": {"col_ret": retn(s["Dref_col"]["probes"][str(last)]["col"], b31["col"], f31["col"]),
                                                 "row_ret": retn(s["Dref_col"]["probes"][str(last)]["row"], b31["row"], f31["row"])}}
    add_table("c_curves", [{"model": "qwen2vl", "arm": n, "readout": b if b < FIN else "final", "ssdc": v, "N": Nb(s[n]["ssdc"], base, flo, b)}
                           for n in s for b, v in enumerate(s[n]["ssdc"])])
    return o

for nm, fn in [("d", analyse_d), ("e", analyse_e), ("c", analyse_c)]:
    try:
        r_ = fn()
        if r_: ANALYSIS[nm] = r_
    except Exception as ex:
        traceback.print_exc(); print(f"analysis of part {nm} failed: {ex!r}")
for part, keys in (("a", A_MODELS), ("b", ["large"])):
    for key in keys:
        try:
            r_ = analyse_scale(part, key)
            if r_: ANALYSIS.setdefault(part, {})[key] = r_
        except Exception as ex:
            traceback.print_exc(); print(f"analysis of part {part} {key} failed: {ex!r}")
for part, val in ANALYSIS.items():
    show(f"PART {part}", val)

## The report for the team: `rb7_report.md`
One section per part: the reviewer's question, what ran, the numbers, the verdict by the fixed rule, and the exact
text for the paper with where it goes. The draft can be edited from this file alone.

In [ ]:
L = []
def md(*lines): L.extend(lines)
def pct(x): return "n/a" if x is None or not np.isfinite(x) else f"{100 * x:.0f}%"
_g_all = []
for f in sorted(OUT_DIR.glob("rb7_gates_session_*.json")): _g_all += json.loads(f.read_text())
_seen = {}
for g in _g_all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
md(f"# RB-7 results — generated {RUN_STAMP}" + ("  (TEST MODE: tiny random models, numbers mean nothing)" if TEST_MODE else ""), "",
   f"Gates: {len(gdf)} checked, {int((gdf.status == 'PASS').sum()) if len(gdf) else 0} pass, **{n_fail} fail**, "
   f"{int((gdf.status == 'SKIP').sum()) if len(gdf) else 0} skipped. Parts in this report: {', '.join(ANALYSIS) or 'none'}.",
   "" if n_fail == 0 else "**Some gates failed: do not use the numbers of a part whose gates failed until it is understood.**", "",
   "Conventions: SSDC normalised N = (arm − floor) / (intact − floor) at that readout; retention of a probe likewise. "
   "Top-1 costs in points on 5,000 ILSVRC images with paired standard errors. Heads are (block, head). "
   "Every verdict is the rule fixed in the notebook before the run (quoted under each part).", "")

# part d
if "d" in ANALYSIS:
    D = ANALYSIS["d"]; cl = [k for k in D_MODELS if k in D and "top1" in D[k]]
    md("## Part d — is the six-head result a block-position effect? (M20, Table 21; 91UE W4, RxJ2 Q2)", "",
       f"Rule: {RULES['D_NULL']}", "", f"Also: {RULES['D_WITHIN']}", "",
       "| model | top-6 heads | top-1 cost of top 6 | block-matched null (mean / 95th / max) | top 6 above | verdict (top-1) | SSDC: top 6 above | within-block ρ (p) |",
       "|---|---|---|---|---|---|---|---|")
    for k in D_MODELS:
        if k not in D: continue
        o = D[k]; t = o.get("top1"); sv = o["ssdc"]; wv = o.get("ssdc_vs_top1")
        heads = "; ".join(f"b{b}: {v}" for b, v in sorted(o["top6"].items(), key=lambda kv: int(kv[0])))
        md(f"| {MODEL_SPECS[k]['label']} | {heads} | " + (f"{t['top6_cost']:.2f} ± {t['top6_se']:.2f}" if t else "—") + " | " +
           (f"{t['null_mean']:.2f} / {t['null_p95']:.2f} / {t['null_max']:.2f}" if t else "—") + " | " +
           (f"{t['n_below']} of {t['n']}" if t else "—") + " | " + (t["verdict"] if t else "—") + " | " +
           f"{sv['n_below']} of {sv['n']} ({sv['verdict']}) | " + (f"{wv['within_block_mean_rho']:.2f} ({wv['within_block_p']:.3f})" if wv else "—") + " |")
    md("")
    if cl:
        vs = [D[k]["top1"]["verdict"] for k in cl]; costs = [D[k]["top1"]["top6_cost"] for k in cl]; nmax = max(D[k]["top1"]["null_max"] for k in cl)
        nb = [D[k]["top1"]["n_below"] for k in cl]; nn = D[cl[0]]["top1"]["n"]
        rw = [D[k]["ssdc_vs_top1"]["within_block_mean_rho"] for k in cl]; pw = max(D[k]["ssdc_vs_top1"]["within_block_p"] for k in cl)
        tb = "; ".join(f"{MODEL_SPECS[k]['label']} blocks {min(map(int, D[k]['top6']))}–{max(map(int, D[k]['top6']))}" for k in cl)
        if all(v == "HOLDS" for v in vs):
            s20 = (f"the six top SSDC heads together cost {min(costs):.1f}–{max(costs):.1f} points, more than each of {nn} random sets of six "
                   f"heads drawn from the same blocks (at most {nmax:.1f}; Appendix H.2)")
        elif all(v != "DOES NOT HOLD" for v in vs):
            s20 = (f"the six top SSDC heads together cost {min(costs):.1f}–{max(costs):.1f} points, more than {min(nb)}–{max(nb)} of {nn} "
                   f"random sets of six heads drawn from the same blocks (Appendix H.2)")
        else:
            s20 = (f"the six top SSDC heads together cost {min(costs):.1f}–{max(costs):.1f} points, but random sets of six heads drawn from "
                   f"the same blocks cost up to {nmax:.1f} (the top six exceed {min(nb)}–{max(nb)} of {nn}), so much of their combined "
                   "cost reflects where they sit, in the first blocks after the window (Appendices F.5 and H.2)")
        md("**Text for the paper.**", "",
           f"- M20 (Section 4.5): replace “and the six top SSDC heads together cost more than each of 24 random sets of one head per block "
           f"(Appendix H.2)” with “and {s20}”.",
           f"- M20 and M11: after “A head’s SSDC cost predicts its top-1 cost (Spearman ρ = 0.55–0.67” add “; within blocks ρ = "
           f"{min(rw):.2f}–{max(rw):.2f}, p ≤ {max(pw, 1 / (D_PERM_N + 1)):.3f}” (in M11 the sentence reads “SSDC cost predicts it better than probe cost does (Spearman ρ = 0.55–0.67 against 0.19–0.33”: add the same clause after 0.55–0.67). (This answers the objection that both costs "
           "simply fall with depth.)",
           f"- Table 21, column “Top 6 vs random”: give, per model, the top-6 cost and “{nn} block-matched sets: mean / max”, from the table above; "
           "keep RB-3's one-per-block comparison in the caption only.",
           f"- Appendix H.2, add: “Because the six top heads sit in the first blocks after the window ({tb}) while random one-head-per-block "
           f"sets spread over blocks 6–11, we also drew {nn} random sets of six heads with exactly the top six's block counts” followed by the M20 result.", "")
    if "DINOv3" in D:
        md(f"DINOv3 (SSDC only, no classifier): top 6 above {D['DINOv3']['ssdc']['n_below']} of {D['DINOv3']['ssdc']['n']} block-matched sets "
           f"({D['DINOv3']['ssdc']['verdict']}); reported in Appendix H.2 only.", "")

# part a
if "a" in ANALYSIS:
    A = ANALYSIS["a"]
    md("## Part a — 320×320 (Appendix H.3; 91UE Q4 and W5)", "", f"Rules: {RULES['AXIS']} {RULES['REBUILD']} {RULES['A_KEEPS']}", "",
       "| model | axis test (block 4) | rebuild N, block 11 (224 → 320) | named head | top-1 rank, 224 → 320 | cost 224 → 320 (pts) | axis 224 → 320 | verdict | ρ per-head cost 224 vs 320 | first-two-block share 224 → 320 | ρ SSDC vs top-1 at 320 (within blocks) |",
       "|---|---|---|---|---|---|---|---|---|---|---|")
    for k in A_MODELS:
        if k not in A or "ref_head" not in A[k]: continue
        o = A[k]; r = o["ref_head"]; at = o["axis_test"]; sv = o.get("ssdc_vs_top1", {})
        md(f"| {MODEL_SPECS[k]['label']} | {at['verdict']} (row off: row {pct(at['rowoff_all']['row_ret'])}, col {pct(at['rowoff_all']['col_ret'])}; "
           f"col off: col {pct(at['coloff_all']['col_ret'])}, row {pct(at['coloff_all']['row_ret'])}) | {f2(o['rebuild_224'])} → {f2(o['rebuild']['Cref'])} | "
           f"({r['head'][0]},{r['head'][1]}) | {r['rank_224']} → {r['rank_320']} of {r['of']} | {f2(r['cost_224'])} → {f2(r['cost_320'])} ± {f2(r['se_320'])} | "
           f"{r['label_224']} → {r['label_320']} | {r['verdict']} | {f2(o['top1_224_vs_320_rho'])} | {pct(o['share_top1_first_two_blocks_224'])} → "
           f"{pct(o['share_top1_first_two_blocks'])} | {f2(sv.get('pooled_rho'))} ({f2(sv.get('within_block_mean_rho'))}) |")
    md("")
    ks = [k for k in A_MODELS if k in A and "ref_head" in A[k]]
    if ks:
        keep_r = [k for k in ks if A[k]["ref_head"]["keeps_rank"]]; keep_a = [k for k in ks if A[k]["ref_head"]["keeps_axis"]]
        heads_txt = ", ".join(f"({A[k]['ref_head']['head'][0]},{A[k]['ref_head']['head'][1]}) is {A[k]['ref_head']['rank_320']} of "
                              f"{A[k]['ref_head']['of']} ({f2(A[k]['ref_head']['cost_320'])} ± {f2(A[k]['ref_head']['se_320'])} points)" for k in ks)
        nr = [A[k]["rebuild"]["Cref"] for k in ks]; n224 = [A[k]["rebuild_224"] for k in ks]
        rho = [A[k]["top1_224_vs_320_rho"] for k in ks]; sh = [A[k]["share_top1_first_two_blocks"] for k in ks]
        axv = [A[k]["axis_test"]["verdict"] for k in ks]
        md("**Text for the paper (Appendix H.3, after “That run also covered 320×320 input, where the top-rebuilder index group costs 7.84, "
           "6.08 and 7.80 points after the window.”, i.e. the sentence as edited by A15):**", "",
           f"“We repeated the single-head test at 320×320 (a 20×20 grid) on the same 5,000 images. "
           f"{'Removing one axis’s planes in every block still removes that axis and spares the other at block 4' if all(v == 'AXES SEPARATE' for v in axv) else 'The axis test at block 4 gives ' + ', '.join(axv).lower()}, "
           f"and after the 0–5 window normalised SSDC entering block 11 recovers to {min(nr):.2f}–{max(nr):.2f} (224: {min(n224):.2f}–{max(n224):.2f}). "
           f"By top-1 cost, {heads_txt}; "
           + (f"all {len(ks)} stay among the three costliest single heads" if len(keep_r) == len(ks) else f"{len(keep_r)} of {len(ks)} stay among the three costliest")
           + (f" and keep the axis they favour at 224 ({', '.join(sorted(set(A[k]['ref_head']['label_224'] for k in ks)))})" if len(keep_a) == len(ks)
              else f", and {len(keep_a)} of {len(ks)} keep the axis they favour at 224")
           + f". Per-head top-1 costs at the two resolutions correlate at ρ = {min(rho):.2f}–{max(rho):.2f}, and the first two blocks after the "
           f"window carry {100 * min(sh):.0f}–{100 * max(sh):.0f}% of the summed cost.”", "",
           "Replace the existing sentence “At 320×320, SSDC loses sensitivity …” only if the SSDC-vs-top-1 ρ at 320 in the table is clearly "
           "below 224's (0.55–0.67); otherwise keep it as is.", "")

# part b
if "b" in ANALYSIS and "large" in ANALYSIS["b"]:
    o = ANALYSIS["b"]["large"]; at = o["axis_test"]; rb = o["rebuild"]
    md("## Part b — naver ViT-L/16 (new Appendix H.4; 91UE W5, RxJ2 W2 scale)", "",
       f"Rules: {RULES['AXIS']} {RULES['REBUILD']} {RULES['B_HEADS']}", "",
       "| test | result | verdict |", "|---|---|---|",
       f"| axis test at block 8 | row off: row {pct(at['rowoff_all']['row_ret'])}, col {pct(at['rowoff_all']['col_ret'])}; col off: col "
       f"{pct(at['coloff_all']['col_ret'])}, row {pct(at['coloff_all']['row_ret'])}; 16 random planes: row {pct(at['randoff_all']['row_ret'])}, "
       f"col {pct(at['randoff_all']['col_ret'])} | {at['verdict']} |",
       f"| rebuild entering block 23 (N) | 0–5: {f2(rb.get('w0_5'))}; 0–11: {f2(rb['Cref'])}; 0–17: {f2(rb.get('w0_17'))} | {rb['verdict']} (main window 0–11) |")
    if "top_ssdc_head" in o and o["top_ssdc_head"]:
        md(f"| top SSDC single head (blocks 12–22) | ({o['top_ssdc_head'][0]},{o['top_ssdc_head'][1]}); first two blocks after the window: {o['top_ssdc_head_in_first_two_blocks']} | — |")
    if "costliest_top1_head" in o:
        c = o["costliest_top1_head"]; sv = o["ssdc_vs_top1"]
        md(f"| costliest single head by top-1 | ({c[0]},{c[1]}) at {c[2]:.2f} ± {c[3]:.2f} points; first two blocks: {o['costliest_top1_head_in_first_two_blocks']} | — |",
           f"| share of summed top-1 cost in blocks 12–13 | {pct(o['share_top1_first_two_blocks'])} | — |",
           f"| SSDC cost vs top-1 cost, 176 heads in blocks 12–22 | ρ = {f2(sv['pooled_rho'])}; within blocks {f2(sv['within_block_mean_rho'])} (p = {f2(sv['within_block_p'], 3)}) | — |")
    am = o.get("axis_map", {})
    if am:
        md(f"| axis map (top heads) | {am['n_above_noise']} of {am['n_mapped']} mapped heads above noise; {am['n_one_axis']} of them favour one axis | — |")
    if "top1_refs" in o:
        t = o["top1_refs"]; md(f"| top-1 intact / window / floor | {t['baseline']:.2f} / {t['window']:.2f} / {t['floor']:.2f} | — |")
    md("", "**Text for the paper (new Appendix H.4, “A larger model”):**", "",
       f"“naver ViT-L/16 has 24 blocks of 16 heads and the same axial RoPE. "
       + ("Removing the row or the column planes in every block removes that axis at block 8 and spares the other. " if at["verdict"] == "AXES SEPARATE"
          else f"The axis test at block 8 reads {at['verdict'].lower()}. ")
       + f"With rotations withheld from blocks 0–11 and restored, normalised SSDC entering block 23 recovers to {f2(rb['Cref'])} "
       f"(0–5: {f2(rb.get('w0_5'))}; 0–17: {f2(rb.get('w0_17'))}). "
       + (f"The costliest single head after the window is ({o['costliest_top1_head'][0]},{o['costliest_top1_head'][1]}), "
          f"{o['costliest_top1_head'][2]:.2f} ± {o['costliest_top1_head'][3]:.2f} points; blocks 12–13 carry {pct(o['share_top1_first_two_blocks'])} "
          f"of the summed single-head cost, and a head’s SSDC cost and its top-1 cost correlate at ρ = {f2(o['ssdc_vs_top1']['pooled_rho'])} "
          f"(within blocks ρ = {f2(o['ssdc_vs_top1']['within_block_mean_rho'])}, p = {f2(o['ssdc_vs_top1']['within_block_p'], 3)}). " if "costliest_top1_head" in o else "")
       + (f"Of the {am['n_above_noise']} mapped heads above noise, {am['n_one_axis']} favour one axis.”" if am else "”"), "")

# part e
if "e" in ANALYSIS:
    E = ANALYSIS["e"]
    if E.get("E1"):
        e1 = E["E1"]
        md("## Part e1 — do the axes compete for attention? (Appendix E.1, A5; NpkG Q2)", "", f"Rule: {RULES['E1']}", "",
           "| model | axis removed (all blocks) | widening along that axis, blocks 1–4 (patches) | other axis's probe vs intact at block 4 (W5, pts) |",
           "|---|---|---|---|")
        for c in e1["cases"]:
            md(f"| {MODEL_SPECS[c['model']]['label']} | {c['removed_axis']} | {c['widening_pts']:+.3f} | {f2(c['other_probe_rise_pts'])} |")
        md("", f"Widening in {e1['n_widening']} of {e1['n_cases']} cases; Spearman(widening, rise) = {f2(e1['spearman_widening_vs_rise'])}. "
               f"**Verdict: {e1['verdict']}.**", "")
        txt = {"WIDER POOLING SUPPORTED": "Removing one axis’s planes widens attention along that axis in blocks 1–4 (Appendix E.1), and the "
                                          "larger the widening, the larger the rise of the other axis’s probe: without that axis’s rotations, heads pool "
                                          "along it and average away what varies along it, which leaves the other axis sharper.",
               "MIXED": "Removing one axis’s planes widens attention along that axis in most but not all models (Appendix E.1), so attention "
                        "pooling may contribute to the rise, but the data do not establish it.",
               "NOT SUPPORTED": "Removing one axis’s planes does not consistently widen attention along that axis (Appendix E.1), so the rise is "
                                "not explained by wider pooling; we leave its mechanism open."}[e1["verdict"]]
        md(f"**Text for the paper (A5: one sentence at the end of the Appendix E.1 paragraph on the rise):** “{txt}” Add the table above to Appendix E.1.", "")
    if E.get("hybrid"):
        h = E["hybrid"]
        md("## Part e2 — what does the hybrid's classifier get from the rotations? (Appendix G.2, A11; RxJ2 Q3)", "", f"Rule: {RULES['E2']}", "",
           f"Blocks 4–11, mean over heads: removing the rotations changes the mean attention distance by {h['rot_off_dist_increase_mean']:+.2f} "
           f"patches (wider in {h['rot_off_blocks_wider']} of 8 blocks) and the entropy by {h['rot_off_entropy_change_mean']:+.2f} nats; removing "
           f"the embedding changes them by {h['ape_off_dist_change_mean']:+.2f} patches and {h['ape_off_entropy_change_mean']:+.2f} nats. "
           f"**Verdict: {h['verdict']}.**", "")
        if h["verdict"] == "ROTATIONS KEEP LATE ATTENTION LOCAL":
            md("**Text for the paper (Appendix G.2, after the Table 22 sentence):** “Without its rotations the hybrid’s attention in blocks 4–11 "
               f"spreads by {h['rot_off_dist_increase_mean']:.2f} patches on average, while removing the embedding changes it by "
               f"{h['ape_off_dist_change_mean']:+.2f}: the rotations keep late attention local, and erasing the position stored in the residual "
               "stream costs the classifier only 0.4 points (Appendix G.3), so the classifier relies on rotation-local attention rather than on "
               "decodable position.”", "")
        else:
            md("**Text for the paper (Appendix G.2):** “Removing the rotations does not make the hybrid’s late attention measurably less local "
               "(Appendix G.2), so what the classifier takes from the rotations is not attention locality as we measure it; RB-6 shows it is not the "
               "position stored in the residual stream either (erasing it costs 0.4 points).”", "")

# part c
if "c" in ANALYSIS:
    o = ANALYSIS["c"]; at = (o.get("axis_test") or {}).get(str(C_AXIS_READOUT)); rb = o.get("rebuild", {})
    md("## Part c — Qwen2-VL-2B's vision encoder (new Appendix H.5; RxJ2 W2)", "",
       f"Rules: {RULES['C_START']} {RULES['AXIS']} {RULES['REBUILD']}", "", f"Checkpoint {QWEN_ID}, revision {o.get('revision')}.", "",
       "| test | result | verdict |", "|---|---|---|",
       f"| SSDC entering block 0 / peak | {o['ssdc_intact_0']:.3f} / {o['peak_ssdc']['value']:.3f} at block {o['peak_ssdc']['block']} (half of peak first at block {o['first_block_half_peak']}) | {o['start_verdict']} |")
    if at:
        md(f"| axis test at block {C_AXIS_READOUT} | row off: row {pct(at['rowoff_all']['row_ret'])}, col {pct(at['rowoff_all']['col_ret'])}; col off: col "
           f"{pct(at['coloff_all']['col_ret'])}, row {pct(at['coloff_all']['row_ret'])}; 20 random planes: row {pct(at['randoff_all']['row_ret'])}, "
           f"col {pct(at['randoff_all']['col_ret'])} | {at['verdict']} |")
    if rb:
        md(f"| rebuild entering block 31 (N) | 0–7: {f2(rb.get('w0_7'))}; 0–15: {f2(rb.get('w0_15'))}; 0–23: {f2(rb.get('w0_23'))} | {o.get('rebuild_verdict')} (main window 0–15) |")
    if o.get("one_axis_window"):
        ow = o["one_axis_window"]
        md(f"| one axis off in 0–15, probes at block 31 | row off: row {pct(ow['row_off_0_15']['row_ret'])}, col {pct(ow['row_off_0_15']['col_ret'])}; "
           f"col off: col {pct(ow['col_off_0_15']['col_ret'])}, row {pct(ow['col_off_0_15']['row_ret'])} | — |")
    md("", "**Text for the paper (new Appendix H.5, “A RoPE encoder trained for a VLM”):**", "",
       "“Qwen2-VL’s vision encoder (675M parameters, 32 blocks, 14-pixel patches) uses axial 2-D RoPE and no absolute embedding. At 224×224 "
       f"its SSDC starts at {o['ssdc_intact_0']:.2f} entering block 0 and peaks at {o['peak_ssdc']['value']:.2f} in block {o['peak_ssdc']['block']}, "
       "so, as in the RoPE ViTs, attention builds the spatial structure. "
       + (("Removing the row or the column planes in every block removes that axis at block 11 and spares the other. " if at["verdict"] == "AXES SEPARATE"
           else f"The axis test at block 11 reads {at['verdict'].lower()}. ") if at else "")
       + (f"With rotations withheld from blocks 0–15 and restored, normalised SSDC entering block 31 recovers to {f2(rb.get('w0_15'))}." if rb else "")
       + " The mechanism therefore " + ("carries over to a RoPE encoder trained for a VLM." if (at and at["verdict"] == "AXES SEPARATE" and o.get("rebuild_verdict") == "REBUILDS")
                                         else "carries over only in part; see the table.") + "”", "",
       "CLIP and SigLIP encoders use absolute embeddings, so they belong to the APE side.", "")

# summary: M24, coverage map
parts_txt = []
if "a" in ANALYSIS:
    ks = [k for k in A_MODELS if k in ANALYSIS["a"] and "ref_head" in ANALYSIS["a"][k]]
    if ks:
        kr = sum(ANALYSIS["a"][k]["ref_head"]["keeps_rank"] for k in ks); ka = sum(ANALYSIS["a"][k]["ref_head"]["keeps_axis"] for k in ks)
        parts_txt.append("the rebuilding heads keep their rank and axis at 320×320" if (kr == ka == len(ks))
                         else f"at 320×320, {kr} of {len(ks)} named heads stay among the three costliest and {ka} keep their axis")
if "b" in ANALYSIS and "large" in ANALYSIS["b"]:
    ob = ANALYSIS["b"]["large"]
    parts_txt.append(f"in ViT-L the axes {'separate' if ob['axis_test']['verdict'] == 'AXES SEPARATE' else 'separate only partly'} and later blocks "
                     f"{ {'REBUILDS': 'rebuild', 'REBUILDS LESS': 'rebuild less', 'DOES NOT REBUILD': 'do not rebuild'}[ob['rebuild']['verdict']] }")
if "c" in ANALYSIS and ANALYSIS["c"].get("axis_test"):
    oc = ANALYSIS["c"]
    parts_txt.append(f"in Qwen2-VL’s vision encoder the axes {'separate' if oc['axis_verdict'] == 'AXES SEPARATE' else 'separate only partly'} and later blocks "
                     f"{ {'REBUILDS': 'rebuild', 'REBUILDS LESS': 'rebuild less', 'DOES NOT REBUILD': 'do not rebuild'}.get(oc.get('rebuild_verdict'), 'were not tested') }")
if parts_txt:
    md("## Limitations (M24) and the coverage map", "",
       f"**M24, replacing “and extending the analysis to larger models and higher resolutions … would show how far these mechanisms generalise”:** "
       f"“{'; '.join(parts_txt)} (Appendix {', '.join(x for x, p in (('H.3', 'a'), ('H.4', 'b'), ('H.5', 'c')) if p in ANALYSIS)}), but we have not tested dense prediction.”", "",
       "**Coverage map:** 91UE Q4 → Answered (part a). 91UE W5 → Answered for resolution and scale (parts a, b); dense prediction stays a limitation. "
       "RxJ2 W2 → RoPE half answered (part c); CLIP/SigLIP are APE. 91UE W4 / RxJ2 Q2 → M20 now uses the block-matched null (part d). "
       "NpkG Q2 → part e1. RxJ2 Q3 → part e2 + RB-6. Use the verdicts above; where a verdict is negative, the text above already says so.", "")
REPORT_MD = "\n".join(L)
(OUT_DIR / "rb7_report.md").write_text(REPORT_MD)
print(REPORT_MD)

## Gate report and export

In [ ]:
print(f"GATE REPORT  ({len(set(g['session'] for g in _g_all))} session(s), {len(_g_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum()) if len(gdf) else 0} pass, {n_fail} fail, {n_skip} skipped")
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in (gdf[gdf.status == "FAIL"].iterrows() if len(gdf) else []): print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)
complete = {}
for part, keys in (("d", D_MODELS), ("a", A_MODELS), ("b", ["large"]), ("e", E_MODELS), ("c", ["qwen2vl"])):
    if part not in PARTS: continue
    for k in keys:
        f = rfile(part, k); complete[f"{part}:{k}"] = bool(f.exists() and json.loads(f.read_text()).get("complete", part == "e" and f.exists()))
print("complete:", complete)
export = {"notebook": "RB-7", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "environment": ENV, "repo": str(repo), "repo_commit": REPO_COMMIT,
          "config": {**CFG_BASE, "parts": PARTS, "strict": STRICT, "qwen": QWEN_ID}, "rules": RULES, "analysis": ANALYSIS,
          "gates": gdf.to_dict("records"), "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "complete": complete,
                                                            "sessions": sorted(set(g["session"] for g in _g_all))}}
(OUT_DIR / f"rb7_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "rb7_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb7_{name}.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")